# UTS PjBL Kecerdasan Buatan & Sistem Cerdas — TIM 12
## Bot Game AI Connect-Four Berbasis Depth-Limited Minimax & Evaluation Function

- **Institusi:** Institut Digital Ekonomi (IDE) LPKIA Bandung — S1 Teknik Informatika / Sistem Informasi
- **Dosen Pengampu:** Wahyu Nurjaya WK, S.T., M.Kom.
- **Ruang Lingkup & Metode AI:** *Adversarial Search* (*Depth-Limited Minimax* + *Heuristic Evaluation Function* & *Alpha-Beta Pruning*)
- **Demo Web UI (GitHub Pages):** [https://zysrnh.github.io/ConnectFour_UTS/](https://zysrnh.github.io/ConnectFour_UTS/)
- **Repository GitHub:** [https://github.com/zysrnh/ConnectFour_UTS](https://github.com/zysrnh/ConnectFour_UTS)

### Cara Menjalankan di Google Colab
1. Klik **`▶ Jalankan semua`** (menu **Runtime → Run all**) di toolbar atas agar seluruh modul Python, pengujian taktis, benchmark, dan UI interaktif tereksekusi berurutan.
2. Scroll ke sel paling bawah (**Bagian 6B**) untuk memainkan **Arena Retro Connect-Four (Main Menu → Arena Beranimasi → Tabel Analisis Akhir)** langsung di Colab menggunakan **Python Kernel**!

### Identitas TIM 12
| Peran | Nama Mahasiswa | NIM | Tanggung Jawab Utama |
| :--- | :--- | :--- | :--- |
| **Mahasiswa A** | `[Isi Nama Mahasiswa A]` | `[NIM A]` | *AI System Architect / Modeler* (State Space, Eval Function, Minimax) |
| **Mahasiswa B** | `[Isi Nama Mahasiswa B]` | `[NIM B]` | *Lead Developer / Tester* (Arena Interaktif, Benchmark, Web UI) |

---
## Bagian 1: Pemodelan AI & Relevansi dengan Materi Sesi 1–4

Implementasi proyek ini memadukan konsep inti yang telah dipelajari dari **Sesi 1 hingga Sesi 4**:

1. **Arsitektur Sistem Cerdas & Intelligent Agent (Sesi 1):**
   - **Input:** Status papan Connect-Four `6 × 7` saat ini dan aksi kolom pemain.
   - **Knowledge Base:** Aturan kemenangan `4-in-a-row` dan bobot fitur *Evaluation Function* (`EVAL_WEIGHTS`).
   - **Inference Engine:** Algoritma *Depth-Limited Minimax* (serta opsi optimasi *Alpha-Beta Pruning*).
   - **Output:** Kolom terbaik `[0..6]`, skor evaluasi papan, dan jumlah node yang dievaluasi.
   - **Formulasi Agen:** `Percept` berupa posisi papan `6 × 7`; `Action` menjatuhkan bidak ke kolom valid; `Performance Measure` diukur dari keberhasilan menang (`WIN_SCORE`), pemblokiran ancaman lawan, dan efisiensi node.
2. **State Space & Search Tree (Sesi 2):**
   - Ruang keadaan dimodelkan sebagai *Game Tree* dengan *branching factor* `b <= 7` dan kedalaman maksimal permainan `m = 42`.
   - Penelusuran dilakukan secara rekursif berbasis *Depth-First Search* (DFS).
3. **Heuristic Function `h(n)` (Sesi 3):**
   - Ketika batas kedalaman pencarian tercapai pada posisi non-terminal (`depth >= max_depth`), fungsi `evaluate_board()` bertindak sebagai fungsi heuristik untuk menaksir keunggulan papan bagi Bot.
4. **Adversarial Search: Minimax & Alpha-Beta Pruning (Sesi 4):**
   - Permainan bersifat *zero-sum*: **Bot AI (`2` / `X`) sebagai MAX** dan **Manusia (`1` / `O`) sebagai MIN**.
   - **Kenapa pakai Depth-Limited Minimax?** Seperti dibahas pada Pre-Test Sesi 4, Minimax murni memiliki kompleksitas `O(b^m)`. Pada Connect-Four (`b <= 7, m = 42`), pencarian penuh sampai terminal state tidak efisien. Pembatasan kedalaman (`max_depth`) + *Evaluation Function* membuat keputusan cepat dan akurat.
   - **Optimasi Alpha-Beta Pruning:** Memangkas cabang yang tidak memengaruhi hasil akhir saat `alpha >= beta`, didukung *center-first column ordering* `[3, 2, 4, 1, 5, 0, 6]`.

In [ ]:
# Setup Environment & Konstanta Global (Sinkron 1:1 dengan js/ai.js)
import math
import random
import sys
import time
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

ROWS = 6
COLS = 7
WINDOW_LENGTH = 4

BOARD_SIZES = {
    "6x7": (6, 7),
    "7x8": (7, 8),
    "7x9": (7, 9),
}

BLOCKED = -1  # Petak batu rintangan (mode Obstacle)
EMPTY = 0
HUMAN = 1   # Pemain 1 / Manusia (MIN) -> Simbol 'O'
AI = 2      # Pemain 2 / Bot AI (MAX)   -> Simbol 'X'

WIN_SCORE = 100000


def get_column_order(cols=COLS):
    """Kembalikan urutan prioritas kolom dari tengah ke pinggir."""
    order = []
    if cols % 2 == 1:
        mid = cols // 2
        order.append(mid)
        for d in range(1, mid + 1):
            if mid - d >= 0:
                order.append(mid - d)
            if mid + d < cols:
                order.append(mid + d)
    else:
        left = cols // 2 - 1
        right = cols // 2
        for d in range(cols // 2):
            if left - d >= 0:
                order.append(left - d)
            if right + d < cols:
                order.append(right + d)
    return order


COLUMN_ORDER = get_column_order(COLS)  # Prioritas kolom tengah ke pinggir (standar 6x7)

EVAL_WEIGHTS = {
    "WIN": 100000,
    "AI_THREE": 12,
    "AI_TWO": 3,
    "OPP_THREE": -80,
    "OPP_TWO": -2,
    "CENTER_PER_DISC": 4,
    "STAR_BONUS": 6,
}

DIFFICULTY_PRESETS = {
    "Easy": {"max_depth": 0, "use_pruning": False, "label": "Easy (Random Move)"},
    "Medium": {"max_depth": 3, "use_pruning": False, "label": "Medium (Depth-Limited Minimax, depth=3)"},
    "Hard": {"max_depth": 5, "use_pruning": True, "label": "Hard (Depth-Limited + Alpha-Beta, depth=5)"},
}

print("=== CEK ENVIRONMENT PRAKTIKUM AI TIM 12 ===")
print("Versi Python       :", sys.version.split()[0])
print("Matplotlib tersedia:", HAS_MATPLOTLIB)
print("Konfigurasi Papan  :", f"{ROWS} baris x {COLS} kolom (opsional: 7x8 Luas, 7x9 Ekstra)")

---
## Bagian 2: Representasi Papan & Aturan Permainan Connect-Four
Papan direpresentasikan sebagai matriks 2D berukuran standar `6 × 7` (juga mendukung varian lebih luas `7 × 8` Luas & `7 × 9` Ekstra; `row 0` = baris paling atas, `row rows-1` = baris paling bawah). Setiap bidak yang dijatuhkan ke kolom akan jatuh ke baris kosong terbawah akibat gravitasi.

In [ ]:
def create_board(rows=ROWS, cols=COLS):
    """Buat papan kosong berukuran rows x cols (baris 0 = atas, baris terakhir = bawah)."""
    return [[EMPTY for _ in range(cols)] for _ in range(rows)]


def create_special_setup(board_mode="classic", rows=ROWS, cols=COLS):
    """Buat papan awal sesuai mode ('classic', 'obstacle', 'star') dan ukuran rows x cols."""
    board = create_board(rows, cols)
    star_cells = []
    obstacles = []
    center_cols = {cols // 2} if cols % 2 == 1 else {cols // 2 - 1, cols // 2}
    side_cols = [c for c in range(cols) if c not in center_cols]

    if board_mode == "obstacle":
        col_pick_count = 4 if cols >= 8 else 3
        chosen = random.sample(side_cols, col_pick_count)
        spots = [
            (rows - 1, chosen[0]),
            (rows - 1, chosen[1]),
            (rows - 2, chosen[1]),
            (rows - 1, chosen[2]),
        ]
        if cols >= 8:
            spots.append((rows - 1, chosen[3]))
        for r, c in spots:
            board[r][c] = BLOCKED
            obstacles.append([r, c])
    elif board_mode == "star":
        inner_cols = list(range(1, cols - 1))
        star_count = 4 if cols >= 8 else 3
        chosen_cols = random.sample(inner_cols, star_count)
        star_rows = list(range(2, rows))
        for c in chosen_cols:
            r = random.choice(star_rows)
            star_cells.append([r, c])

    return {"board": board, "star_cells": star_cells, "obstacles": obstacles}


def clone_board(board):
    """Salin papan 2D secara mendalam."""
    return [row[:] for row in board]


def print_board(board):
    """Tampilkan papan Connect-Four ke konsol sesuai dimensi papan."""
    rows = len(board)
    cols = len(board[0])
    symbol_map = {BLOCKED: "#", EMPTY: ".", HUMAN: "O", AI: "X"}
    print("  " + " ".join(str(c) for c in range(cols)))
    print(" +" + "--" * cols + "+")
    for r in range(rows):
        row_str = " ".join(symbol_map[board[r][c]] for c in range(cols))
        print(f"{r}| {row_str}|")
    print(" +" + "--" * cols + "+")


def is_valid_location(board, col):
    """Cek apakah kolom valid dan baris paling atas masih kosong."""
    cols = len(board[0])
    return 0 <= col < cols and board[0][col] == EMPTY


def get_valid_locations(board):
    """Kembalikan daftar kolom valid sesuai urutan tengah-ke-pinggir."""
    cols = len(board[0])
    return [col for col in get_column_order(cols) if is_valid_location(board, col)]


def get_next_open_row(board, col):
    """Cari indeks baris terbawah yang masih kosong pada kolom tertentu."""
    for r in range(len(board) - 1, -1, -1):
        if board[r][col] == EMPTY:
            return r
    return -1


def drop_piece(board, row, col, piece):
    """Letakkan bidak pada koordinat (row, col)."""
    board[row][col] = piece


def get_winner_info(board):
    """Cek pemenang (4-in-a-row) atau seri beserta daftar koordinat kemenangan."""
    rows = len(board)
    cols = len(board[0])
    players = (HUMAN, AI)

    # Cek horizontal
    for r in range(rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            p = board[r][c]
            if p in players and p == board[r][c + 1] == board[r][c + 2] == board[r][c + 3]:
                return p, [(r, c), (r, c + 1), (r, c + 2), (r, c + 3)]

    # Cek vertikal
    for c in range(cols):
        for r in range(rows - WINDOW_LENGTH + 1):
            p = board[r][c]
            if p in players and p == board[r + 1][c] == board[r + 2][c] == board[r + 3][c]:
                return p, [(r, c), (r + 1, c), (r + 2, c), (r + 3, c)]

    # Cek diagonal turun ke kanan (\)
    for r in range(rows - WINDOW_LENGTH + 1):
        for c in range(cols - WINDOW_LENGTH + 1):
            p = board[r][c]
            if p in players and p == board[r + 1][c + 1] == board[r + 2][c + 2] == board[r + 3][c + 3]:
                return p, [(r, c), (r + 1, c + 1), (r + 2, c + 2), (r + 3, c + 3)]

    # Cek diagonal naik ke kanan (/)
    for r in range(WINDOW_LENGTH - 1, rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            p = board[r][c]
            if p in players and p == board[r - 1][c + 1] == board[r - 2][c + 2] == board[r - 3][c + 3]:
                return p, [(r, c), (r - 1, c + 1), (r - 2, c + 2), (r - 3, c + 3)]

    # Cek seri jika semua kolom teratas penuh
    if not get_valid_locations(board):
        return "DRAW", []

    return None, []


def check_winner(board):
    """Wrapper ringkas mengembalikan 1 (HUMAN), 2 (AI), 'DRAW', atau None."""
    winner, _ = get_winner_info(board)
    return winner


# Uji cepat papan & cek kemenangan
sample_board = create_board()
for col_idx in [3, 3, 2, 2, 4, 1]:
    r_open = get_next_open_row(sample_board, col_idx)
    drop_piece(sample_board, r_open, col_idx, AI if col_idx != 1 else HUMAN)

print_board(sample_board)
print("Status Game Contoh:", check_winner(sample_board))

---
## Bagian 3: Fungsi Evaluasi Papan (*Heuristic Evaluation Function*)
Karena pencarian dibatasi oleh `max_depth`, Bot AI membutuhkan **Evaluation Function** untuk menilai posisi papan non-terminal:

- Memeriksa seluruh **69 jendela 4-petak** (horizontal, vertikal, diagonal naik & turun):
  - `3 bidak AI + 1 kosong`: **+12**
  - `2 bidak AI + 2 kosong`: **+3**
  - `3 bidak Lawan + 1 kosong`: **-80** (penalti besar agar AI memprioritaskan blokir ancaman langsung)
  - `2 bidak Lawan + 2 kosong`: **-2**
- Memberikan **bonus kontrol kolom tengah (`kolom 3`)** sebesar **+4** per selisih bidak di kolom tengah.

In [ ]:
def evaluate_window(window, ai_piece):
    """Hitung skor heuristik satu jendela berisi 4 petak dari sudut pandang ai_piece."""
    if BLOCKED in window:
        return 0
    opp_piece = HUMAN if ai_piece == AI else AI
    ai_count = window.count(ai_piece)
    opp_count = window.count(opp_piece)
    empty_count = window.count(EMPTY)

    if ai_count == 4:
        return EVAL_WEIGHTS["WIN"]
    if opp_count == 4:
        return -EVAL_WEIGHTS["WIN"]
    if ai_count == 3 and empty_count == 1:
        return EVAL_WEIGHTS["AI_THREE"]
    if ai_count == 2 and empty_count == 2:
        return EVAL_WEIGHTS["AI_TWO"]
    if opp_count == 3 and empty_count == 1:
        return EVAL_WEIGHTS["OPP_THREE"]
    if opp_count == 2 and empty_count == 2:
        return EVAL_WEIGHTS["OPP_TWO"]
    return 0


def analyze_board(board, ai_piece=AI, star_cells=None):
    """Hitung total skor evaluasi papan beserta rincian fitur heuristiknya."""
    rows = len(board)
    cols = len(board[0])
    opp_piece = HUMAN if ai_piece == AI else AI
    center_cols = [cols // 2] if cols % 2 == 1 else [cols // 2 - 1, cols // 2]
    center_weight = EVAL_WEIGHTS["CENTER_PER_DISC"] if cols % 2 == 1 else EVAL_WEIGHTS["CENTER_PER_DISC"] // 2

    center_ai = sum(1 for r in range(rows) for c in center_cols if board[r][c] == ai_piece)
    center_opp = sum(1 for r in range(rows) for c in center_cols if board[r][c] == opp_piece)

    ai_three = 0
    ai_two = 0
    opp_three = 0
    opp_two = 0
    window_score = 0

    def process_window(w):
        nonlocal ai_three, ai_two, opp_three, opp_two, window_score
        if BLOCKED in w:
            return
        ai_c = w.count(ai_piece)
        opp_c = w.count(opp_piece)
        emp_c = w.count(EMPTY)
        if ai_c == 3 and emp_c == 1:
            ai_three += 1
        elif ai_c == 2 and emp_c == 2:
            ai_two += 1
        elif opp_c == 3 and emp_c == 1:
            opp_three += 1
        elif opp_c == 2 and emp_c == 2:
            opp_two += 1
        window_score += evaluate_window(w, ai_piece)

    # Horizontal
    for r in range(rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r][c + 1], board[r][c + 2], board[r][c + 3]])

    # Vertikal
    for c in range(cols):
        for r in range(rows - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r + 1][c], board[r + 2][c], board[r + 3][c]])

    # Diagonal turun (\)
    for r in range(rows - WINDOW_LENGTH + 1):
        for c in range(cols - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r + 1][c + 1], board[r + 2][c + 2], board[r + 3][c + 3]])

    # Diagonal naik (/)
    for r in range(WINDOW_LENGTH - 1, rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r - 1][c + 1], board[r - 2][c + 2], board[r - 3][c + 3]])

    star_ai = 0
    star_opp = 0
    if star_cells:
        for sr, sc in star_cells:
            if 0 <= sr < rows and 0 <= sc < cols:
                if board[sr][sc] == ai_piece:
                    star_ai += 1
                elif board[sr][sc] == opp_piece:
                    star_opp += 1

    center_score = (center_ai - center_opp) * center_weight
    star_score = (star_ai - star_opp) * EVAL_WEIGHTS["STAR_BONUS"]
    return {
        "total_score": window_score + center_score + star_score,
        "center_ai": center_ai,
        "center_opp": center_opp,
        "center_score": center_score,
        "star_ai": star_ai,
        "star_opp": star_opp,
        "star_score": star_score,
        "ai_three": ai_three,
        "ai_two": ai_two,
        "opp_three": opp_three,
        "opp_two": opp_two,
        "window_score": window_score,
    }


def evaluate_board(board, ai_piece=AI, star_cells=None):
    """Fungsi Evaluasi Papan (Evaluation Function) untuk non-terminal state."""
    return analyze_board(board, ai_piece, star_cells)["total_score"]


# Uji fungsi evaluasi pada papan contoh
eval_demo = analyze_board(sample_board, AI)
print("Rincian Evaluasi Papan Contoh (Sudut Pandang AI):")
for k, v in eval_demo.items():
    print(f"  - {k:14s}: {v}")

---
## Bagian 4: Implementasi Depth-Limited Minimax & Alpha-Beta Pruning
Di bawah ini terdapat dua versi algoritma pencarian:
1. `minimax(...)`: **Depth-Limited Minimax Murni** (menjelajah seluruh cabang hingga batas `max_depth`).
2. `minimax_alpha_beta(...)`: **Depth-Limited Minimax + Alpha-Beta Pruning** (memangkas cabang saat `alpha >= beta`).

Keduanya menerapkan **penalti kedalaman** (`WIN_SCORE - depth` saat menang, `depth - WIN_SCORE` saat kalah) seperti pola pada Sesi 4 agar Bot AI memilih kemenangan tercepat dan menunda kekalahan selama mungkin.

In [ ]:
def minimax(board, depth, max_depth, is_maximizing, ai_piece, stats, star_cells=None):
    """Depth-Limited Minimax Murni tanpa Alpha-Beta Pruning."""
    stats["nodes"] += 1
    opp_piece = HUMAN if ai_piece == AI else AI

    winner = check_winner(board)
    if winner == ai_piece:
        return WIN_SCORE - depth
    if winner == opp_piece:
        return depth - WIN_SCORE
    if winner == "DRAW":
        return 0

    if depth >= max_depth:
        return evaluate_board(board, ai_piece, star_cells)

    valid_cols = get_valid_locations(board)

    if is_maximizing:
        best_score = -math.inf
        for col in valid_cols:
            row = get_next_open_row(board, col)
            board[row][col] = ai_piece
            score = minimax(board, depth + 1, max_depth, False, ai_piece, stats, star_cells)
            board[row][col] = EMPTY
            if score > best_score:
                best_score = score
        return best_score

    best_score = math.inf
    for col in valid_cols:
        row = get_next_open_row(board, col)
        board[row][col] = opp_piece
        score = minimax(board, depth + 1, max_depth, True, ai_piece, stats, star_cells)
        board[row][col] = EMPTY
        if score < best_score:
            best_score = score
    return best_score


def minimax_alpha_beta(board, depth, max_depth, alpha, beta, is_maximizing, ai_piece, stats, star_cells=None):
    """Depth-Limited Minimax dengan optimasi Alpha-Beta Pruning."""
    stats["nodes"] += 1
    opp_piece = HUMAN if ai_piece == AI else AI

    winner = check_winner(board)
    if winner == ai_piece:
        return WIN_SCORE - depth
    if winner == opp_piece:
        return depth - WIN_SCORE
    if winner == "DRAW":
        return 0

    if depth >= max_depth:
        return evaluate_board(board, ai_piece, star_cells)

    valid_cols = get_valid_locations(board)

    if is_maximizing:
        max_eval = -math.inf
        for col in valid_cols:
            row = get_next_open_row(board, col)
            board[row][col] = ai_piece
            eval_score = minimax_alpha_beta(
                board, depth + 1, max_depth, alpha, beta, False, ai_piece, stats, star_cells
            )
            board[row][col] = EMPTY
            if eval_score > max_eval:
                max_eval = eval_score
            if max_eval > alpha:
                alpha = max_eval
            if alpha >= beta:
                break  # Pemangkasan cabang MIN
        return max_eval

    min_eval = math.inf
    for col in valid_cols:
        row = get_next_open_row(board, col)
        board[row][col] = opp_piece
        eval_score = minimax_alpha_beta(
            board, depth + 1, max_depth, alpha, beta, True, ai_piece, stats, star_cells
        )
        board[row][col] = EMPTY
        if eval_score < min_eval:
            min_eval = eval_score
        if min_eval < beta:
            beta = min_eval
        if alpha >= beta:
            break  # Pemangkasan cabang MAX
    return min_eval


def find_best_move(board, max_depth=4, ai_piece=AI, use_pruning=True, star_cells=None):
    """Cari kolom terbaik beserta skor evaluasi tiap kolom dan jumlah node."""
    working_board = clone_board(board)
    valid_cols = get_valid_locations(working_board)
    column_scores = {c: None for c in range(len(working_board[0]))}

    if not valid_cols:
        return {
            "best_col": -1,
            "best_score": 0,
            "column_scores": column_scores,
            "nodes": 0,
            "elapsed_ms": 0.0,
            "max_depth": max_depth,
            "use_pruning": use_pruning,
        }

    stats = {"nodes": 0}
    t0 = time.perf_counter()

    best_score = -math.inf
    best_col = valid_cols[0]
    alpha = -math.inf
    beta = math.inf

    for col in valid_cols:
        row = get_next_open_row(working_board, col)
        working_board[row][col] = ai_piece

        if use_pruning:
            score = minimax_alpha_beta(
                working_board, 1, max_depth, alpha, beta, False, ai_piece, stats, star_cells
            )
        else:
            score = minimax(
                working_board, 1, max_depth, False, ai_piece, stats, star_cells
            )

        working_board[row][col] = EMPTY
        column_scores[col] = score

        if score > best_score:
            best_score = score
            best_col = col
        if use_pruning and best_score > alpha:
            alpha = best_score

    t1 = time.perf_counter()
    return {
        "best_col": best_col,
        "best_score": best_score,
        "column_scores": column_scores,
        "nodes": stats["nodes"],
        "elapsed_ms": round((t1 - t0) * 1000, 2),
        "max_depth": max_depth,
        "use_pruning": use_pruning,
    }


def choose_move(board, difficulty="Hard", ai_piece=AI, use_pruning=None, custom_depth=None, star_cells=None):
    """Pilih langkah Bot AI sesuai tingkat kesulitan ('Easy', 'Medium', 'Hard')."""
    preset = DIFFICULTY_PRESETS.get(difficulty, DIFFICULTY_PRESETS["Hard"])

    if difficulty == "Easy" and custom_depth is None:
        valid_cols = get_valid_locations(board)
        column_scores = {c: None for c in range(len(board[0]))}
        if not valid_cols:
            return {
                "best_col": -1, "best_score": 0, "column_scores": column_scores,
                "nodes": 0, "elapsed_ms": 0.0, "max_depth": 0,
                "use_pruning": False, "is_random": True,
            }
        working_board = clone_board(board)
        t0 = time.perf_counter()
        for col in valid_cols:
            row = get_next_open_row(working_board, col)
            working_board[row][col] = ai_piece
            column_scores[col] = evaluate_board(working_board, ai_piece, star_cells)
            working_board[row][col] = EMPTY
        chosen = random.choice(valid_cols)
        t1 = time.perf_counter()
        return {
            "best_col": chosen,
            "best_score": column_scores[chosen],
            "column_scores": column_scores,
            "nodes": len(valid_cols),
            "elapsed_ms": round((t1 - t0) * 1000, 2),
            "max_depth": 0,
            "use_pruning": False,
            "is_random": True,
        }

    max_depth = custom_depth if custom_depth is not None else preset["max_depth"]
    pruning = use_pruning if use_pruning is not None else preset["use_pruning"]
    res = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=pruning, star_cells=star_cells)
    res["is_random"] = False
    return res


def compare_algorithms(board, max_depth=4, ai_piece=AI, star_cells=None):
    """Bandingkan Minimax Murni vs Alpha-Beta Pruning pada state papan yang sama."""
    pure = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=False, star_cells=star_cells)
    ab = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=True, star_cells=star_cells)
    reduction = round(((pure["nodes"] - ab["nodes"]) / pure["nodes"]) * 100, 1) if pure["nodes"] > 0 else 0.0
    return {
        "max_depth": max_depth,
        "pure": pure,
        "alphabeta": ab,
        "reduction_pct": reduction,
        "same_decision": pure["best_col"] == ab["best_col"],
    }

---
## Bagian 5: Pengujian Terstruktur & Benchmark Efisiensi Node
Sel-sel di bawah ini berjalan otomatis (tanpa `input()`) untuk menguji:
1. Ketepatan taktis Bot AI (menang 1 langkah, memblokir lawan, memilih kolom tengah di awal).
2. Perbandingan jumlah *node* yang dievaluasi dan waktu komputasi **Minimax Murni vs Alpha-Beta Pruning** pada berbagai kedalaman (`depth = 1..4`).
3. Simulasi pertandingan antar tingkat kesulitan (5 game per pasangan, mengikuti standar uji coba Sesi 4).

In [ ]:
# 1. Pengujian Taktis Unit Test
def run_tactical_tests():
    # Kasus A: Papan kosong -> harus pilih kolom tengah (3)
    b_empty = create_board()
    res_empty = find_best_move(b_empty, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_empty["best_col"] == 3, f"Harusnya kolom 3, dapat {res_empty['best_col']}"

    # Kasus B: AI punya 3 beruntun (kolom 0,1,2 di baris 5) -> wajib selesaikan menang di kolom 3
    b_win = create_board()
    for c in [0, 1, 2]:
        drop_piece(b_win, 5, c, AI)
    for c in [0, 1]:
        drop_piece(b_win, 4, c, HUMAN)
    res_win = find_best_move(b_win, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_win["best_col"] == 3 and res_win["best_score"] == WIN_SCORE - 1

    # Kasus C: Manusia punya 3 beruntun (kolom 4,5,6 di baris 5) -> Bot wajib blokir di kolom 3
    b_block = create_board()
    for c in [4, 5, 6]:
        drop_piece(b_block, 5, c, HUMAN)
    drop_piece(b_block, 5, 0, AI)
    drop_piece(b_block, 4, 0, AI)
    res_block = find_best_move(b_block, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_block["best_col"] == 3, f"Harusnya blokir kolom 3, dapat {res_block['best_col']}"

    # Kasus D: Keputusan Minimax Murni dan Alpha-Beta harus 100% identik
    comp = compare_algorithms(b_block, max_depth=4, ai_piece=AI)
    assert comp["same_decision"], "Keputusan Minimax vs Alpha-Beta berbeda!"

    # Kasus E: Mode Obstacle (BLOCKED = -1) menonaktifkan 4-in-a-row melewati batu
    b_obs = create_board()
    b_obs[5][2] = BLOCKED
    drop_piece(b_obs, 5, 0, AI)
    drop_piece(b_obs, 5, 1, AI)
    drop_piece(b_obs, 5, 3, AI)
    assert check_winner(b_obs) is None

    # Kasus F: Mode Star memberi bonus skor evaluasi +6 per petak bintang
    b_star = create_board()
    drop_piece(b_star, 5, 2, AI)
    assert evaluate_board(b_star, AI, [[5, 2]]) - evaluate_board(b_star, AI, []) == EVAL_WEIGHTS["STAR_BONUS"]

    # Kasus G: Ukuran papan lebih luas (7x8 & 7x9) mendukung pusat kolom & menang di pinggir kanan
    b_wide = create_board(7, 9)
    res_wide = find_best_move(b_wide, max_depth=2, ai_piece=AI, use_pruning=True)
    assert res_wide["best_col"] == 4, f"Papan 7x9 kosong harusnya kolom tengah 4, dapat {res_wide['best_col']}"
    for c in [5, 6, 7, 8]:
        drop_piece(b_wide, 6, c, AI)
    assert check_winner(b_wide) == AI

    print("[PASS] Semua 7 pengujian taktis (termasuk Obstacle, Star Tiles, & Papan 7x9) berhasil 100%!")
    print(f"  - Papan kosong -> Kol {res_empty['best_col']} (skor {res_empty['best_score']})")
    print(f"  - Deteksi menang 1 langkah -> Kol {res_win['best_col']} (skor {res_win['best_score']})")
    print(f"  - Blokir ancaman lawan     -> Kol {res_block['best_col']} (skor {res_block['best_score']})")
    print(f"  - Konsistensi Minimax vs Alpha-Beta (d=4): Kol {comp['pure']['best_col']} == Kol {comp['alphabeta']['best_col']} "
          f"(hemat {comp['reduction_pct']}% node)")


run_tactical_tests()

In [ ]:
# 2. Benchmark Minimax Murni vs Alpha-Beta Pruning (Depth 1 s.d. 4)
bench_board = create_board()
sample_opening = [(3, HUMAN), (3, AI), (2, HUMAN), (4, AI), (2, HUMAN), (3, AI)]
for col_m, piece_m in sample_opening:
    drop_piece(bench_board, get_next_open_row(bench_board, col_m), col_m, piece_m)

print("Posisi Papan Benchmark (6 langkah pembuka):")
print_board(bench_board)

rows_bench = []
for d in [1, 2, 3, 4]:
    comp = compare_algorithms(bench_board, max_depth=d, ai_piece=AI)
    rows_bench.append({
        "Max Depth": d,
        "Kolom Dipilih": comp["alphabeta"]["best_col"],
        "Skor Eval": comp["alphabeta"]["best_score"],
        "Node Minimax": comp["pure"]["nodes"],
        "Node Alpha-Beta": comp["alphabeta"]["nodes"],
        "Penghematan Node (%)": comp["reduction_pct"],
        "Waktu Minimax (ms)": comp["pure"]["elapsed_ms"],
        "Waktu Alpha-Beta (ms)": comp["alphabeta"]["elapsed_ms"],
        "Keputusan Sama?": comp["same_decision"],
    })

df_bench = pd.DataFrame(rows_bench)
print("\n=== TABEL BENCHMARK EFISIENSI NODE ===")
print(df_bench.to_string(index=False))

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    depths = df_bench["Max Depth"]

    axes[0].bar(depths - 0.18, df_bench["Node Minimax"], width=0.35, label="Minimax Murni", color="#dc2626")
    axes[0].bar(depths + 0.18, df_bench["Node Alpha-Beta"], width=0.35, label="Alpha-Beta Pruning", color="#1d4ed8")
    axes[0].set_title("Perbandingan Jumlah Node Dievaluasi")
    axes[0].set_xlabel("Max Depth")
    axes[0].set_ylabel("Jumlah Node")
    axes[0].set_xticks(depths)
    axes[0].legend()
    axes[0].grid(axis="y", linestyle="--", alpha=0.4)

    axes[1].bar(depths, df_bench["Penghematan Node (%)"], width=0.45, color="#15803d")
    axes[1].set_title("Persentase Penghematan Node Alpha-Beta")
    axes[1].set_xlabel("Max Depth")
    axes[1].set_ylabel("Penghematan (%)")
    axes[1].set_ylim(0, 100)
    axes[1].set_xticks(depths)
    axes[1].grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

In [ ]:
# 3. Simulasi Turnamen Otomatis Bot vs Bot (5 Pertandingan per Skenario)
def simulate_match(p1_cfg, p2_cfg, seed=None, opening=None):
    if seed is not None:
        random.seed(seed)
    board = create_board()
    moves = 0
    nodes_p1, nodes_p2 = [], []

    if opening:
        for idx, col in enumerate(opening):
            piece = HUMAN if idx % 2 == 0 else AI
            drop_piece(board, get_next_open_row(board, col), col, piece)
            moves += 1
        turn = HUMAN if len(opening) % 2 == 0 else AI
    else:
        turn = HUMAN  # HUMAN = Pemain 1, AI = Pemain 2

    while True:
        cfg = p1_cfg if turn == HUMAN else p2_cfg
        res = choose_move(
            board,
            difficulty=cfg["difficulty"],
            ai_piece=turn,
            use_pruning=cfg.get("use_pruning"),
            custom_depth=cfg.get("custom_depth"),
        )
        col = res["best_col"]
        drop_piece(board, get_next_open_row(board, col), col, turn)
        moves += 1
        if turn == HUMAN:
            nodes_p1.append(res["nodes"])
        else:
            nodes_p2.append(res["nodes"])

        winner = check_winner(board)
        if winner is not None:
            return {
                "winner": winner,
                "moves": moves,
                "avg_nodes_p1": round(sum(nodes_p1) / max(1, len(nodes_p1)), 1),
                "avg_nodes_p2": round(sum(nodes_p2) / max(1, len(nodes_p2)), 1),
            }
        turn = AI if turn == HUMAN else HUMAN


varied_openings = [(3, 3), (2, 3), (4, 3), (3, 2), (3, 4)]

scenarios = [
    ("Easy (Random) vs Medium (Depth 3 Minimax)", {"difficulty": "Easy"}, {"difficulty": "Medium"}, None),
    ("Easy (Random) vs Hard (Depth 5 Alpha-Beta)", {"difficulty": "Easy"}, {"difficulty": "Hard"}, None),
    ("Medium (Depth 3) vs Hard (Depth 5 Alpha-Beta)", {"difficulty": "Medium"}, {"difficulty": "Hard"}, varied_openings),
]

summary_rows = []
for name, p1_c, p2_c, openings in scenarios:
    p1_wins = p2_wins = draws = 0
    total_moves = []
    avg_n2 = []
    for idx in range(5):
        op = openings[idx] if openings else None
        out = simulate_match(p1_c, p2_c, seed=100 + idx, opening=op)
        total_moves.append(out["moves"])
        avg_n2.append(out["avg_nodes_p2"])
        if out["winner"] == HUMAN:
            p1_wins += 1
        elif out["winner"] == AI:
            p2_wins += 1
        else:
            draws += 1

    summary_rows.append({
        "Skenario (P1 vs P2)": name,
        "P1 Menang": p1_wins,
        "P2 Menang": p2_wins,
        "Seri": draws,
        "Rata-rata Langkah": round(sum(total_moves) / len(total_moves), 1),
        "Rata-rata Node P2": round(sum(avg_n2) / len(avg_n2), 1),
    })

df_sim = pd.DataFrame(summary_rows)
print("=== HASIL SIMULASI 5 GAME PER SKENARIO ===")
print(df_sim.to_string(index=False))

---
## Bagian 6A: Arena Teks Konsol Cadangan (`play_game`)
Mode teks konsol Python murni jika ingin bermain lewat terminal/input teks. Ubah `JALANKAN_GAME_INTERAKTIF = True` bila ingin memainkannya lewat `input()`.

In [ ]:
def play_game(difficulty="Hard", human_first=True, custom_depth=None, use_pruning=None, board_size="6x7"):
    """Arena interaktif Manusia ('O') vs Bot AI ('X') mode konsol."""
    rows, cols = BOARD_SIZES.get(board_size, (ROWS, COLS))
    board = create_board(rows, cols)
    current_turn = HUMAN if human_first else AI
    step = 0

    print("=== ARENA INTERAKTIF CONNECT-FOUR AI (TIM 12) ===")
    print(f"Ukuran Papan      : {rows}x{cols}")
    print(f"Tingkat Kesulitan : {difficulty}")
    print("Simbol Pemain     : Manusia = 'O' | Bot AI = 'X'")
    print(f"Masukkan nomor kolom 0 s.d. {cols - 1} (atau 'q' untuk keluar).\n")

    while True:
        print_board(board)
        print(f"Skor Evaluasi Papan Statis (sudut pandang AI): {evaluate_board(board, AI)}")

        if current_turn == HUMAN:
            valid = [c for c in range(cols) if is_valid_location(board, c)]
            while True:
                raw = input(f"[Giliran Manusia 'O'] Pilih kolom {valid} (atau 'q'): ").strip()
                if raw.lower() == "q":
                    print("Permainan dihentikan oleh pemain.")
                    return
                if raw.isdigit() and int(raw) in valid:
                    col = int(raw)
                    break
                print("Input tidak valid! Pilih kolom yang masih tersedia.")

            row = get_next_open_row(board, col)
            drop_piece(board, row, col, HUMAN)
            step += 1
            print(f"-> Manusia menjatuhkan bidak di Kolom {col}, Baris {row}.\n")
        else:
            print("[Giliran Bot AI 'X'] Menghitung langkah terbaik...")
            res = choose_move(
                board,
                difficulty=difficulty,
                ai_piece=AI,
                use_pruning=use_pruning,
                custom_depth=custom_depth,
            )
            col = res["best_col"]
            row = get_next_open_row(board, col)
            drop_piece(board, row, col, AI)
            step += 1

            scores_str = " | ".join(
                f"K{c}:{res['column_scores'][c]}" if res["column_scores"][c] is not None else f"K{c}:Penuh"
                for c in range(cols)
            )
            print(f"-> Bot memilih Kolom {col} (Baris {row}) | Skor: {res['best_score']} "
                  f"| Node: {res['nodes']} | Waktu: {res['elapsed_ms']} ms")
            print(f"   Skor Evaluasi Kolom: [{scores_str}]\n")

        winner, win_cells = get_winner_info(board)
        if winner is not None:
            print_board(board)
            if winner == "DRAW":
                print(f"=== HASIL AKHIR: SERI setelah {step} langkah! ===")
            elif winner == HUMAN:
                print(f"=== SELAMAT! Manusia ('O') MENANG dalam {step} langkah! Koordinat: {win_cells} ===")
            else:
                print(f"=== BOT AI ('X') MENANG dalam {step} langkah! Koordinat: {win_cells} ===")
            break

        current_turn = AI if current_turn == HUMAN else HUMAN


# Ubah menjadi True jika ingin bermain lewat konsol teks
JALANKAN_GAME_INTERAKTIF = False

if JALANKAN_GAME_INTERAKTIF:
    play_game(difficulty="Hard", human_first=True)
else:
    print("Mode konsol standby (aman untuk Run All). Scroll ke sel paling bawah (Bagian 6B) untuk UI Retro Interaktif!")

---
## Bagian 6B: Arena UI Retro 3-Layar Interaktif di Google Colab (`Python Kernel` + Web UI Identik)
Sel terakhir ini merender **UI Retro Connect-Four 3-Layar** (*Main Menu → Arena Bersih Beranimasi → Tabel Analisis Pasca-Main*) yang **sama persis dengan Demo GitHub Pages**, sekaligus menghubungkan setiap langkah Bot AI & Benchmark langsung ke fungsi **Python (`choose_move` & `compare_algorithms`)** di kernel Google Colab!

> **Penting:** Pastikan semua sel di atas sudah dijalankan (**`Runtime → Run all` / `▶ Jalankan semua`**) sebelum memainkan sel ini.

In [ ]:
import json

RETRO_UI_HTML = "<style>\n:root {\n  /* Retro Color Palette (from reference) */\n  --retro-yellow: #ffc567;\n  --retro-pink: #fb7da8;\n  --retro-coral: #fd5a46;\n  --retro-purple: #552cb7;\n  --retro-green: #00995e;\n  --retro-blue: #058cd7;\n\n  /* Neutrals */\n  --ink: #111111;\n  --white: #ffffff;\n  --cream: #fff8eb;\n  --slot-empty: #fff8eb;\n\n  /* Semantic tokens */\n  --p1-color: var(--retro-coral);\n  --p2-color: var(--retro-yellow);\n  --board-bg: var(--retro-blue);\n\n  --shadow-sm: 2px 2px 0 var(--ink);\n  --shadow-md: 4px 4px 0 var(--ink);\n  --shadow-lg: 6px 6px 0 var(--ink);\n\n  --radius: 8px;\n  --radius-sm: 4px;\n  --font-sans: \"Trebuchet MS\", \"Segoe UI\", Roboto, Helvetica, Arial, sans-serif;\n  --font-mono: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace;\n}\n\n* {\n  box-sizing: border-box;\n  margin: 0;\n  padding: 0;\n}\n\nbody {\n  font-family: var(--font-sans);\n  color: var(--ink);\n  line-height: 1.4;\n  min-height: 100dvh;\n  display: flex;\n  flex-direction: column;\n  background-color: var(--retro-yellow);\n  /* Pola grid retro tanpa CSS gradient */\n  background-image: url(\"data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='32' height='32'%3E%3Cpath d='M 32 0 L 0 0 0 32' fill='none' stroke='%23111111' stroke-opacity='0.11' stroke-width='1'/%3E%3C/svg%3E\");\n}\n\n/* Top Navigation Bar */\n.topbar {\n  background-color: var(--white);\n  border-bottom: 3px solid var(--ink);\n  flex-shrink: 0;\n}\n\n.topbar-inner {\n  max-width: 1280px;\n  margin: 0 auto;\n  padding: 9px 18px;\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 10px;\n}\n\n.brand-title {\n  font-size: 17px;\n  font-weight: 900;\n  letter-spacing: -0.01em;\n}\n\n.topbar-actions {\n  display: flex;\n  gap: 8px;\n}\n\n/* Tactile Retro Buttons */\n.btn {\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  gap: 7px;\n  padding: 8px 13px;\n  font-family: var(--font-sans);\n  font-size: 13px;\n  font-weight: 800;\n  border-radius: var(--radius-sm);\n  border: 2px solid var(--ink);\n  background-color: var(--white);\n  color: var(--ink);\n  box-shadow: var(--shadow-sm);\n  cursor: pointer;\n  text-decoration: none;\n  transition: transform 0.08s ease, box-shadow 0.08s ease, background-color 0.1s ease;\n}\n\n.btn:hover:not(:disabled) {\n  transform: translate(-1px, -1px);\n  box-shadow: 3px 3px 0 var(--ink);\n}\n\n.btn:active:not(:disabled) {\n  transform: translate(2px, 2px);\n  box-shadow: 0 0 0 var(--ink);\n}\n\n.btn:disabled {\n  opacity: 0.42;\n  cursor: not-allowed;\n  box-shadow: none;\n}\n\n.btn-primary { background-color: var(--retro-purple); color: var(--white); }\n.btn-green { background-color: var(--retro-green); color: var(--white); }\n.btn-coral { background-color: var(--retro-coral); color: var(--ink); }\n.btn-pink { background-color: var(--retro-pink); color: var(--ink); }\n.btn-colab { background-color: var(--retro-yellow); color: var(--ink); }\n.btn-dark { background-color: var(--retro-purple); color: var(--white); }\n.btn-sm { padding: 5px 9px; font-size: 12px; }\n.btn-lg { padding: 11px 18px; font-size: 15px; letter-spacing: 0.03em; }\n\n.btn-icon {\n  width: 17px;\n  height: 17px;\n  flex-shrink: 0;\n  transition: transform 0.14s ease;\n}\n\n.home-play-btn:hover .btn-icon {\n  transform: translateX(3px) scale(1.1);\n}\n\n/* View Shells (Fit 1-screen centered) */\n.view-shell {\n  width: 100%;\n  max-width: 1280px;\n  margin: 0 auto;\n  padding: 18px 16px 14px;\n  flex: 1;\n}\n\n.view-centered {\n  display: flex;\n  align-items: center;\n  justify-content: center;\n}\n\n/* Retro Cards */\n.card {\n  position: relative;\n  background-color: var(--white);\n  border: 3px solid var(--ink);\n  border-radius: var(--radius);\n  box-shadow: var(--shadow-lg);\n  padding: 14px;\n}\n\n.card-hero {\n  padding: 0;\n  overflow: visible;\n}\n\n.floating-pill {\n  position: absolute;\n  top: -14px;\n  left: 50%;\n  transform: translateX(-50%);\n  background-color: var(--white);\n  color: var(--ink);\n  border: 2.5px solid var(--ink);\n  border-radius: 999px;\n  padding: 2px 14px;\n  font-family: var(--font-mono);\n  font-size: 11px;\n  font-weight: 800;\n  letter-spacing: 0.04em;\n  white-space: nowrap;\n  box-shadow: var(--shadow-sm);\n  z-index: 2;\n}\n\n/* VIEW 1: Main Menu Utama */\n.home-stage {\n  width: 100%;\n  max-width: 440px;\n  margin: 0 auto;\n}\n\n.retro-hero-head {\n  background-color: var(--retro-coral);\n  border-bottom: 3px solid var(--ink);\n  border-top-left-radius: 5px;\n  border-top-right-radius: 5px;\n  padding: 22px 14px 14px;\n  display: flex;\n  align-items: center;\n  justify-content: center;\n  gap: 12px;\n}\n\n.retro-stripes {\n  flex: 1;\n  max-width: 72px;\n  display: flex;\n  flex-direction: column;\n  gap: 4px;\n}\n\n.retro-stripes span {\n  display: block;\n  height: 2.5px;\n  background-color: var(--ink);\n}\n\n.retro-wordmark {\n  display: inline-flex;\n  align-items: center;\n  gap: 2px;\n  font-size: 25px;\n  font-weight: 900;\n  letter-spacing: 0.02em;\n  line-height: 1;\n  user-select: none;\n}\n\n.retro-wordmark span {\n  display: inline-block;\n  -webkit-text-stroke: 1.5px var(--ink);\n  text-shadow: 2px 2px 0 var(--ink);\n}\n\n.rw-1 { color: var(--retro-yellow); }\n.rw-2 { color: var(--retro-pink); }\n.rw-3 { color: var(--retro-green); }\n.rw-4 { color: var(--retro-blue); }\n.rw-5 { color: var(--white); }\n.rw-6 { color: var(--retro-yellow); }\n.rw-7 { color: var(--retro-pink); }\n.rw-num {\n  margin-left: 6px;\n  padding: 2px 8px;\n  font-size: 19px;\n  background-color: var(--retro-yellow);\n  color: var(--ink);\n  border: 2.5px solid var(--ink);\n  border-radius: var(--radius-sm);\n  box-shadow: 2px 2px 0 var(--ink);\n  -webkit-text-stroke: 0;\n  text-shadow: none;\n}\n\n.hero-body {\n  padding: 14px;\n  background-color: var(--white);\n  border-bottom-left-radius: 5px;\n  border-bottom-right-radius: 5px;\n}\n\n.home-body {\n  display: flex;\n  flex-direction: column;\n  gap: 14px;\n}\n\n.showcase-wrap {\n  background-color: var(--cream);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  padding: 12px;\n  display: flex;\n  flex-direction: column;\n  align-items: center;\n  gap: 8px;\n}\n\n.showcase-board {\n  background-color: var(--retro-blue);\n  border: 2.5px solid var(--ink);\n  border-radius: var(--radius-sm);\n  box-shadow: 3px 3px 0 var(--ink);\n  padding: 7px;\n  display: grid;\n  grid-template-columns: repeat(5, 24px);\n  gap: 6px;\n  transition: transform 0.16s ease;\n}\n\n.showcase-wrap:hover .showcase-board {\n  transform: translateY(-2px);\n}\n\n.sc-cell {\n  width: 24px;\n  height: 24px;\n  border-radius: 50%;\n  background-color: var(--cream);\n  border: 2px solid var(--ink);\n}\n\n.sc-cell.sc-y { background-color: var(--retro-yellow); }\n.sc-cell.sc-c { background-color: var(--retro-coral); }\n\n.sc-cell.sc-win {\n  outline: 2.5px solid var(--retro-green);\n  outline-offset: 1px;\n  animation: winPulse 0.8s ease-in-out infinite alternate;\n}\n\n.showcase-tagline {\n  font-size: 12px;\n  font-weight: 800;\n  text-align: center;\n  color: var(--ink);\n}\n\n.home-menu-buttons {\n  display: flex;\n  flex-direction: column;\n  gap: 10px;\n}\n\n.home-play-btn {\n  width: 100%;\n  padding: 13px 18px;\n  font-size: 16px;\n}\n\n.home-sub-buttons {\n  display: grid;\n  grid-template-columns: repeat(2, minmax(0, 1fr));\n  gap: 10px;\n}\n\n/* VIEW 2: Menu Setup Pertandingan */\n.setup-stage {\n  width: 100%;\n  max-width: 530px;\n  margin: 0 auto;\n}\n\n.setup-head {\n  background-color: var(--retro-coral);\n  border-bottom: 3px solid var(--ink);\n  border-top-left-radius: 5px;\n  border-top-right-radius: 5px;\n  padding: 18px 12px 10px;\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 8px;\n}\n\n.setup-head-title {\n  font-size: 13px;\n  font-weight: 900;\n  text-transform: uppercase;\n  letter-spacing: 0.02em;\n}\n\n.setup-body {\n  padding: 12px 14px;\n}\n\n.menu-section {\n  margin-bottom: 10px;\n}\n\n.label-row-split {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 8px;\n  margin-bottom: 6px;\n}\n\n.menu-label {\n  display: block;\n  font-size: 11px;\n  font-weight: 900;\n  text-transform: uppercase;\n  letter-spacing: 0.04em;\n  margin-bottom: 5px;\n}\n\n.shape-toggle {\n  display: inline-flex;\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  overflow: hidden;\n  background-color: var(--cream);\n  box-shadow: 1.5px 1.5px 0 var(--ink);\n}\n\n.shape-btn {\n  display: inline-flex;\n  align-items: center;\n  gap: 4px;\n  padding: 3px 8px;\n  font-family: var(--font-sans);\n  font-size: 11px;\n  font-weight: 800;\n  border: none;\n  background-color: transparent;\n  color: var(--ink);\n  cursor: pointer;\n  transition: background-color 0.1s ease;\n}\n\n.shape-btn + .shape-btn {\n  border-left: 2px solid var(--ink);\n}\n\n.shape-btn.active {\n  background-color: var(--retro-pink);\n}\n\n.shape-ico {\n  width: 10px;\n  height: 10px;\n  border: 1.5px solid var(--ink);\n  background-color: var(--retro-yellow);\n  display: inline-block;\n}\n\n.shape-ico.circle { border-radius: 50%; }\n.shape-ico.square { border-radius: 2px; }\n\n.board-size-bar {\n  margin-top: 8px;\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 8px;\n}\n\n.size-label {\n  font-size: 11px;\n  font-weight: 800;\n  color: #374151;\n}\n\n.size-toggle {\n  display: inline-grid;\n  grid-template-columns: repeat(3, auto);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  overflow: hidden;\n  background-color: var(--cream);\n  box-shadow: 1.5px 1.5px 0 var(--ink);\n}\n\n.size-btn {\n  padding: 4px 9px;\n  font-family: var(--font-mono);\n  font-size: 11px;\n  font-weight: 800;\n  border: none;\n  background-color: transparent;\n  color: var(--ink);\n  cursor: pointer;\n  transition: background-color 0.1s ease;\n}\n\n.size-btn + .size-btn {\n  border-left: 2px solid var(--ink);\n}\n\n.size-btn.active {\n  background-color: var(--retro-yellow);\n}\n\n.choice-grid {\n  display: grid;\n  gap: 8px;\n}\n\n.cols-2-always { grid-template-columns: repeat(2, minmax(0, 1fr)); }\n.cols-3-always { grid-template-columns: repeat(3, minmax(0, 1fr)); }\n\n.choice-card {\n  text-align: left;\n  padding: 7px 9px;\n  background-color: var(--cream);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  box-shadow: var(--shadow-sm);\n  cursor: pointer;\n  display: flex;\n  flex-direction: column;\n  gap: 2px;\n  font-family: var(--font-sans);\n  color: var(--ink);\n  transition: transform 0.08s ease, box-shadow 0.08s ease, background-color 0.1s ease;\n}\n\n.choice-card strong {\n  font-size: 12.5px;\n  font-weight: 900;\n  line-height: 1.2;\n}\n\n.choice-card span {\n  font-size: 10.5px;\n  font-weight: 600;\n  color: #374151;\n  line-height: 1.2;\n}\n\n.choice-card:hover {\n  transform: translate(-1px, -1px);\n  box-shadow: 3px 3px 0 var(--ink);\n  background-color: var(--white);\n}\n\n.choice-card.active {\n  background-color: var(--retro-yellow);\n  box-shadow: 3px 3px 0 var(--ink);\n}\n\n.choice-card.just-picked {\n  animation: cardPickPop 0.22s ease-out;\n}\n\n@keyframes cardPickPop {\n  0% { transform: scale(0.96); }\n  55% { transform: scale(1.03); }\n  100% { transform: scale(1); }\n}\n\n.compact-row-card {\n  flex-direction: row;\n  align-items: center;\n  gap: 8px;\n}\n\n.tile-mode-card {\n  align-items: center;\n  text-align: center;\n  padding: 7px 6px;\n}\n\n.tile-badge {\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  width: 24px;\n  height: 24px;\n  border: 2px solid var(--ink);\n  border-radius: 4px;\n  font-family: var(--font-mono);\n  font-size: 12px;\n  font-weight: 900;\n  margin-bottom: 1px;\n}\n\n.tile-badge.blue { background-color: var(--retro-blue); color: var(--white); }\n.tile-badge.stone { background-color: var(--retro-purple); color: var(--white); }\n.tile-badge.star { background-color: var(--retro-coral); color: var(--retro-yellow); font-size: 14px; }\n\n/* Visual Tokens in Setup Cards */\n.vc-icon-row {\n  display: flex;\n  align-items: center;\n  gap: 4px;\n  flex-shrink: 0;\n}\n\n.mini-disc {\n  width: 15px;\n  height: 15px;\n  border-radius: 50%;\n  border: 2px solid var(--ink);\n  display: inline-block;\n}\n\n.mini-disc.coral { background-color: var(--retro-coral); }\n\n.bot-chip {\n  font-family: var(--font-mono);\n  font-size: 9px;\n  font-weight: 900;\n  padding: 1px 4px;\n  border: 1.5px solid var(--ink);\n  border-radius: 3px;\n  color: var(--ink);\n}\n\n.bot-chip.coral { background-color: var(--retro-coral); }\n.bot-chip.yellow { background-color: var(--retro-yellow); }\n\n.vs-tag {\n  font-family: var(--font-mono);\n  font-size: 9px;\n  font-weight: 900;\n}\n\n.diff-top {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 6px;\n  width: 100%;\n}\n\n.power-bars {\n  display: inline-flex;\n  align-items: flex-end;\n  gap: 2px;\n}\n\n.power-bars .bar {\n  width: 5px;\n  height: 9px;\n  border: 1.5px solid var(--ink);\n  border-radius: 1px;\n  background-color: var(--white);\n  transform-origin: bottom;\n}\n\n.power-bars .bar:nth-child(2) { height: 12px; }\n.power-bars .bar:nth-child(3) { height: 15px; }\n.power-bars .bar.on.green { background-color: var(--retro-green); }\n.power-bars .bar.on.yellow { background-color: var(--retro-yellow); }\n.power-bars .bar.on.coral { background-color: var(--retro-coral); }\n\n.choice-card.active .power-bars .bar.on {\n  animation: barRise 0.25s ease-out;\n}\n\n@keyframes barRise {\n  0% { transform: scaleY(0.45); }\n  70% { transform: scaleY(1.15); }\n  100% { transform: scaleY(1); }\n}\n\n.custom-chip {\n  font-family: var(--font-mono);\n  font-size: 9px;\n  font-weight: 900;\n  background-color: var(--retro-pink);\n  border: 1.5px solid var(--ink);\n  border-radius: 3px;\n  padding: 1px 5px;\n}\n\n.first-card {\n  flex-direction: row;\n  align-items: center;\n  gap: 7px;\n  padding: 7px 8px;\n}\n\n.first-disc {\n  width: 21px;\n  height: 21px;\n  border-radius: 50%;\n  border: 2px solid var(--ink);\n  flex-shrink: 0;\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  font-family: var(--font-mono);\n  font-weight: 900;\n  font-size: 12px;\n}\n\n.first-disc.coral { background-color: var(--retro-coral); }\n.first-disc.yellow { background-color: var(--retro-yellow); }\n.first-disc.random { background-color: var(--retro-pink); color: var(--ink); }\n\n.custom-box {\n  background-color: var(--retro-pink);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  padding: 7px 10px;\n  box-shadow: var(--shadow-sm);\n}\n\n.controls-grid {\n  display: grid;\n  gap: 10px;\n}\n\n.control-group {\n  display: flex;\n  flex-direction: column;\n  gap: 3px;\n}\n\n.control-group label {\n  font-size: 11px;\n  font-weight: 800;\n}\n\n.control-group select,\n.mini-select {\n  padding: 5px 8px;\n  font-size: 12px;\n  font-weight: 700;\n  font-family: var(--font-sans);\n  color: var(--ink);\n  background-color: var(--cream);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n}\n\n.setup-actions {\n  margin-top: 10px;\n  padding-top: 10px;\n  border-top: 2px solid var(--ink);\n}\n\n.setup-actions .btn {\n  width: 100%;\n}\n\n/* VIEW 3 & 4: Arena Layout */\n.arena-shell {\n  display: grid;\n  grid-template-columns: minmax(0, 560px);\n  justify-content: center;\n  gap: 20px;\n  align-items: start;\n}\n\n.arena-shell.has-analysis {\n  grid-template-columns: minmax(0, 1fr) minmax(0, 1fr);\n  justify-content: stretch;\n}\n\n@media (max-width: 960px) {\n  .arena-shell.has-analysis {\n    grid-template-columns: 1fr;\n  }\n}\n\n.arena-topbar {\n  background-color: var(--retro-coral);\n  border-bottom: 3px solid var(--ink);\n  border-top-left-radius: 5px;\n  border-top-right-radius: 5px;\n  padding: 18px 10px 10px;\n  display: flex;\n  flex-wrap: wrap;\n  align-items: center;\n  justify-content: space-between;\n  gap: 6px;\n}\n\n.arena-nav-btns,\n.arena-badges,\n.arena-quick-btns {\n  display: flex;\n  align-items: center;\n  gap: 5px;\n}\n\n.arena-bvb-bar {\n  background-color: var(--retro-yellow);\n  border-bottom: 3px solid var(--ink);\n  padding: 7px 12px;\n  display: flex;\n  align-items: center;\n  justify-content: center;\n  gap: 10px;\n}\n\n.arena-body {\n  padding: 12px;\n}\n\n.card-header {\n  display: flex;\n  flex-wrap: wrap;\n  align-items: center;\n  justify-content: space-between;\n  gap: 10px;\n  margin-bottom: 12px;\n  padding-bottom: 10px;\n  border-bottom: 2.5px solid var(--ink);\n}\n\n.card-title {\n  font-size: 14px;\n  font-weight: 900;\n  text-transform: uppercase;\n  letter-spacing: 0.04em;\n}\n\n.badge {\n  display: inline-block;\n  font-family: var(--font-mono);\n  font-size: 11px;\n  font-weight: 800;\n  padding: 3px 7px;\n  border-radius: var(--radius-sm);\n  background-color: var(--cream);\n  border: 2px solid var(--ink);\n  color: var(--ink);\n}\n\n.badge-accent { background-color: var(--retro-yellow); }\n.badge-pink { background-color: var(--retro-pink); }\n\n/* Status Banner */\n.status-banner {\n  display: flex;\n  flex-wrap: wrap;\n  align-items: center;\n  justify-content: space-between;\n  gap: 8px;\n  padding: 8px 11px;\n  margin-bottom: 8px;\n  border-radius: var(--radius-sm);\n  border: 2.5px solid var(--ink);\n  box-shadow: var(--shadow-sm);\n  background-color: var(--cream);\n  font-weight: 800;\n  font-size: 13px;\n  transition: background-color 0.15s ease, transform 0.15s ease;\n}\n\n.status-banner.turn-p1 { background-color: var(--retro-pink); color: var(--ink); }\n.status-banner.turn-p2 { background-color: var(--retro-yellow); color: var(--ink); }\n.status-banner.game-win { background-color: var(--retro-green); color: var(--white); }\n.status-banner.game-draw { background-color: var(--cream); color: var(--ink); }\n\n.status-left {\n  display: flex;\n  align-items: center;\n  gap: 8px;\n}\n\n.status-dot {\n  width: 14px;\n  height: 14px;\n  border-radius: 50%;\n  border: 2px solid var(--ink);\n  flex-shrink: 0;\n}\n\n.status-dot.p1 { background-color: var(--p1-color); }\n.status-dot.p2 { background-color: var(--p2-color); }\n.status-dot.neutral { background-color: var(--white); }\n.status-dot.shape-square { border-radius: 3px; }\n\n.thinking-dots {\n  display: none;\n  align-items: center;\n  gap: 4px;\n}\n\n.thinking-dots.active { display: inline-flex; }\n\n.thinking-dots span {\n  width: 5px;\n  height: 5px;\n  border-radius: 50%;\n  background-color: var(--ink);\n  animation: dotBounce 0.65s infinite ease-in-out;\n}\n\n.thinking-dots span:nth-child(2) { animation-delay: 0.12s; }\n.thinking-dots span:nth-child(3) { animation-delay: 0.24s; }\n\n@keyframes dotBounce {\n  0%, 80%, 100% { transform: translateY(0); opacity: 0.4; }\n  40% { transform: translateY(-4px); opacity: 1; }\n}\n\n/* Connect-Four Board (Fit viewport smoothly) */\n.board-wrapper {\n  display: flex;\n  flex-direction: column;\n  align-items: center;\n  gap: 6px;\n}\n\n.hover-preview-row {\n  width: 100%;\n  max-width: min(100%, 510px, calc(48dvh * var(--board-cols, 7) / var(--board-rows, 6)));\n  height: 28px;\n  display: grid;\n  grid-template-columns: repeat(var(--board-cols, 7), 1fr);\n  gap: 6px;\n  padding: 0 8px;\n  align-items: center;\n  justify-items: center;\n}\n\n.ghost-slot {\n  width: 24px;\n  height: 24px;\n  display: flex;\n  align-items: center;\n  justify-content: center;\n}\n\n.ghost-disc {\n  width: 22px;\n  height: 22px;\n  border-radius: 50%;\n  background-color: var(--p1-color);\n  border: 2px solid var(--ink);\n  box-shadow: 2px 2px 0 var(--ink);\n  opacity: 0;\n  transform: translateY(4px);\n  transition: opacity 0.1s ease, transform 0.1s ease;\n}\n\n.ghost-disc.shape-square {\n  border-radius: 5px;\n}\n\n.ghost-disc.visible {\n  opacity: 1;\n  animation: ghostBob 0.75s infinite ease-in-out;\n}\n\n@keyframes ghostBob {\n  0%, 100% { transform: translateY(0); }\n  50% { transform: translateY(-4px); }\n}\n\n.col-selectors {\n  width: 100%;\n  max-width: min(100%, 510px, calc(48dvh * var(--board-cols, 7) / var(--board-rows, 6)));\n  display: grid;\n  grid-template-columns: repeat(var(--board-cols, 7), 1fr);\n  gap: 6px;\n  padding: 0 8px;\n}\n\n.col-btn {\n  height: 30px;\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  background-color: var(--cream);\n  color: var(--ink);\n  font-family: var(--font-mono);\n  font-size: 11px;\n  font-weight: 800;\n  box-shadow: 2px 2px 0 var(--ink);\n  cursor: pointer;\n  display: flex;\n  align-items: center;\n  justify-content: center;\n  transition: transform 0.08s ease, box-shadow 0.08s ease, background-color 0.08s ease;\n}\n\n.col-btn:hover:not(:disabled),\n.col-btn.col-hover:not(:disabled) {\n  background-color: var(--retro-pink);\n  transform: translate(-1px, -1px);\n  box-shadow: 3px 3px 0 var(--ink);\n}\n\n.col-btn:active:not(:disabled) {\n  transform: translate(2px, 2px);\n  box-shadow: 0 0 0 var(--ink);\n}\n\n.col-btn.col-best { background-color: var(--retro-yellow); }\n\n.col-btn:disabled {\n  opacity: 0.35;\n  cursor: not-allowed;\n  box-shadow: none;\n}\n\n.board-grid {\n  width: 100%;\n  max-width: min(100%, 510px, calc(48dvh * var(--board-cols, 7) / var(--board-rows, 6)));\n  background-color: var(--board-bg);\n  border: 3px solid var(--ink);\n  border-radius: var(--radius);\n  box-shadow: var(--shadow-md);\n  padding: 8px;\n  display: grid;\n  grid-template-columns: repeat(var(--board-cols, 7), 1fr);\n  gap: 6px;\n  overflow: hidden;\n}\n\n.cell {\n  aspect-ratio: 1 / 1;\n  background-color: var(--slot-empty);\n  border-radius: 50%;\n  border: 2px solid var(--ink);\n  position: relative;\n  cursor: pointer;\n  display: flex;\n  align-items: center;\n  justify-content: center;\n  transition: background-color 0.1s ease;\n}\n\n.board-grid.shape-square .cell,\n.board-grid.shape-square .disc {\n  border-radius: 7px;\n}\n\n.cell.col-active-hover { background-color: #fde68a; }\n.cell.target-landing { background-color: var(--retro-pink); }\n.cell.disabled { cursor: default; }\n\n/* Obstacle Tile (BLOCKED = -1) */\n.cell.is-blocked {\n  background-color: var(--retro-purple);\n  border-radius: 6px;\n  cursor: not-allowed;\n}\n\n.cell.is-blocked::before {\n  content: \"✕\";\n  color: var(--white);\n  font-family: var(--font-mono);\n  font-weight: 900;\n  font-size: 16px;\n  line-height: 1;\n}\n\n/* Star Bonus Tile */\n.cell.is-star {\n  background-color: #fef08a;\n}\n\n.cell.is-star::before {\n  content: \"★\";\n  position: absolute;\n  color: var(--retro-coral);\n  font-size: 16px;\n  line-height: 1;\n  pointer-events: none;\n  animation: starTwinkle 1.4s ease-in-out infinite alternate;\n}\n\n.cell.is-star.has-disc::before {\n  top: 2px;\n  right: 3px;\n  font-size: 10px;\n  color: var(--ink);\n  background-color: var(--white);\n  border: 1px solid var(--ink);\n  border-radius: 3px;\n  padding: 0 2px;\n  z-index: 2;\n  animation: none;\n}\n\n@keyframes starTwinkle {\n  from { transform: scale(0.9); }\n  to { transform: scale(1.12); }\n}\n\n.disc {\n  width: 100%;\n  height: 100%;\n  border-radius: 50%;\n  position: relative;\n  display: flex;\n  align-items: center;\n  justify-content: center;\n}\n\n.disc.p1 { background-color: var(--p1-color); border: 2px solid var(--ink); }\n.disc.p2 { background-color: var(--p2-color); border: 2px solid var(--ink); }\n\n.board-grid.shape-square .disc::before {\n  content: \"\";\n  width: 44%;\n  height: 44%;\n  border: 2px solid var(--ink);\n  border-radius: 2px;\n  background-color: rgba(255, 255, 255, 0.35);\n}\n\n.disc.drop-anim {\n  animation: discDropDynamic calc(140ms + var(--drop-rows, 3) * 38ms) cubic-bezier(0.22, 0.72, 0.36, 1) forwards;\n}\n\n@keyframes discDropDynamic {\n  0% { transform: translateY(calc(var(--drop-rows, 5) * -112%)); }\n  76% { transform: translateY(5%); }\n  88% { transform: translateY(-5%); }\n  100% { transform: translateY(0); }\n}\n\n.cell.last-move::after {\n  content: \"\";\n  position: absolute;\n  width: 9px;\n  height: 9px;\n  border-radius: 50%;\n  background-color: var(--white);\n  border: 2px solid var(--ink);\n  pointer-events: none;\n  z-index: 3;\n}\n\n.cell.win-cell {\n  outline: 3.5px solid var(--retro-green);\n  outline-offset: 2px;\n  animation: winPulse 0.75s ease-in-out infinite alternate;\n  animation-delay: calc(var(--win-idx, 0) * 110ms);\n}\n\n@keyframes winPulse {\n  from { transform: scale(1); }\n  to { transform: scale(1.07); }\n}\n\n.board-legend {\n  width: 100%;\n  max-width: 440px;\n  display: flex;\n  flex-wrap: wrap;\n  justify-content: center;\n  gap: 18px;\n  font-size: 12px;\n  font-weight: 800;\n  padding-top: 4px;\n}\n\n.legend-item {\n  display: inline-flex;\n  align-items: center;\n  gap: 6px;\n}\n\n.legend-swatch {\n  width: 13px;\n  height: 13px;\n  border-radius: 50%;\n  border: 2px solid var(--ink);\n  display: inline-block;\n}\n\n.legend-swatch.shape-square { border-radius: 3px; }\n.legend-swatch.p1 { background-color: var(--p1-color); }\n.legend-swatch.p2 { background-color: var(--p2-color); }\n\n/* Post-Game Panel & Table */\n.postgame-banner {\n  margin-top: 8px;\n  margin-bottom: 14px;\n  padding: 12px 14px;\n  border: 2.5px solid var(--ink);\n  border-radius: var(--radius-sm);\n  box-shadow: var(--shadow-sm);\n  background-color: var(--retro-green);\n  color: var(--white);\n  display: flex;\n  flex-wrap: wrap;\n  align-items: center;\n  justify-content: space-between;\n  gap: 10px;\n}\n\n.postgame-banner.is-bot-win {\n  background-color: var(--retro-coral);\n  color: var(--ink);\n}\n\n.postgame-banner.is-draw {\n  background-color: var(--cream);\n  color: var(--ink);\n}\n\n.postgame-title { font-size: 16px; font-weight: 900; }\n.postgame-sub { font-size: 12px; font-weight: 700; opacity: 0.95; }\n\n.postgame-cta {\n  display: flex;\n  gap: 8px;\n  flex-wrap: wrap;\n}\n\n.metrics-grid {\n  display: grid;\n  grid-template-columns: repeat(4, minmax(0, 1fr));\n  gap: 8px;\n  margin-bottom: 14px;\n}\n\n@media (max-width: 560px) {\n  .metrics-grid { grid-template-columns: repeat(2, minmax(0, 1fr)); }\n}\n\n.metric-box {\n  background-color: var(--cream);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  box-shadow: var(--shadow-sm);\n  padding: 8px 10px;\n}\n\n.metric-box.accent-yellow { background-color: var(--retro-yellow); }\n.metric-box.accent-pink { background-color: var(--retro-pink); }\n\n.metric-label {\n  font-size: 10px;\n  font-weight: 800;\n  text-transform: uppercase;\n  letter-spacing: 0.03em;\n}\n\n.metric-value {\n  margin-top: 2px;\n  font-family: var(--font-mono);\n  font-size: 15px;\n  font-weight: 900;\n}\n\n.section-label {\n  font-size: 12px;\n  font-weight: 900;\n  margin-bottom: 8px;\n  text-transform: uppercase;\n  letter-spacing: 0.04em;\n}\n\n.analysis-table-wrap {\n  max-height: 210px;\n  overflow-y: auto;\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  background-color: var(--cream);\n}\n\n.analysis-table {\n  width: 100%;\n  border-collapse: collapse;\n  font-size: 12px;\n  font-family: var(--font-mono);\n}\n\n.analysis-table th,\n.analysis-table td {\n  padding: 6px 8px;\n  text-align: left;\n  border-bottom: 1.5px solid var(--ink);\n  white-space: nowrap;\n}\n\n.analysis-table thead th {\n  position: sticky;\n  top: 0;\n  background-color: var(--retro-yellow);\n  font-family: var(--font-sans);\n  font-weight: 900;\n  z-index: 1;\n}\n\n.analysis-table tbody tr {\n  animation: rowSlideIn 0.2s ease-out both;\n  animation-delay: calc(var(--row-idx, 0) * 18ms);\n}\n\n@keyframes rowSlideIn {\n  from { opacity: 0; transform: translateX(-6px); }\n  to { opacity: 1; transform: translateX(0); }\n}\n\n.analysis-table tbody tr.row-bot { background-color: var(--white); }\n.analysis-table tbody tr.row-win { background-color: #bbf7d0; font-weight: 800; }\n\n/* Figgle-Style Scoring Legend Card */\n.scoring-legend-card {\n  background-color: var(--retro-coral);\n  border: 2.5px solid var(--ink);\n  border-radius: var(--radius-sm);\n  padding: 12px;\n  box-shadow: var(--shadow-sm);\n}\n\n.scoring-legend-head {\n  display: flex;\n  justify-content: space-between;\n  align-items: center;\n  padding-bottom: 8px;\n  margin-bottom: 10px;\n  border-bottom: 2px solid var(--ink);\n  font-weight: 900;\n  font-size: 13px;\n  text-transform: uppercase;\n}\n\n.scoring-legend-list {\n  display: flex;\n  flex-direction: column;\n  gap: 7px;\n}\n\n.legend-score-row {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 8px;\n  font-size: 12px;\n  font-weight: 800;\n}\n\n.legend-pattern {\n  display: flex;\n  align-items: center;\n  gap: 8px;\n}\n\n.mini-tiles {\n  display: inline-flex;\n  gap: 3px;\n}\n\n.mini-tile {\n  width: 21px;\n  height: 21px;\n  background-color: var(--white);\n  border: 2px solid var(--ink);\n  border-radius: 4px;\n  display: flex;\n  align-items: center;\n  justify-content: center;\n}\n\n.mini-dot {\n  width: 11px;\n  height: 11px;\n  border-radius: 50%;\n  border: 1.5px solid var(--ink);\n}\n\n.mini-dot.ai { background-color: var(--retro-yellow); }\n.mini-dot.opp { background-color: var(--retro-coral); }\n.mini-dot.empty { background-color: var(--cream); border-style: dashed; }\n.mini-dot.center { background-color: var(--retro-blue); }\n.mini-dot.star { background-color: var(--retro-yellow); border-radius: 2px; }\n\n.legend-right {\n  display: flex;\n  align-items: center;\n  gap: 6px;\n  font-family: var(--font-mono);\n}\n\n.legend-live-count {\n  background-color: var(--white);\n  border: 1.5px solid var(--ink);\n  border-radius: var(--radius-sm);\n  padding: 1px 6px;\n  font-size: 11px;\n}\n\n.legend-points {\n  min-width: 56px;\n  text-align: right;\n  font-size: 13px;\n  font-weight: 900;\n}\n\n/* Compare Card */\n.compare-caption {\n  font-size: 12px;\n  font-weight: 600;\n  margin-bottom: 8px;\n}\n\n.compare-result {\n  padding: 10px 12px;\n  background-color: var(--cream);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  font-size: 13px;\n}\n\n.compare-table {\n  width: 100%;\n  border-collapse: collapse;\n  font-size: 12px;\n  font-family: var(--font-mono);\n}\n\n.compare-table th,\n.compare-table td {\n  text-align: left;\n  padding: 6px 8px;\n  border-bottom: 1.5px solid var(--ink);\n}\n\n.compare-table th { font-family: var(--font-sans); font-weight: 900; }\n\n.compare-summary {\n  margin-top: 8px;\n  padding: 8px 10px;\n  background-color: var(--retro-green);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  color: var(--white);\n  font-weight: 800;\n  font-size: 12px;\n}\n\n/* Modals with Open & Close Animations */\n.modal-backdrop {\n  position: fixed;\n  inset: 0;\n  background-color: rgba(17, 17, 17, 0.58);\n  display: flex;\n  align-items: center;\n  justify-content: center;\n  padding: 16px;\n  z-index: 50;\n  overflow-y: auto;\n  animation: fadeInBg 0.18s ease-out forwards;\n}\n\n.modal-backdrop.is-closing {\n  animation: fadeOutBg 0.18s ease-in forwards;\n}\n\n.modal-card {\n  width: 100%;\n  max-width: 520px;\n  animation: popIn 0.2s cubic-bezier(0.22, 1, 0.36, 1) forwards;\n}\n\n.modal-backdrop.is-closing .modal-card {\n  animation: popOut 0.18s ease-in forwards;\n}\n\n@keyframes fadeInBg {\n  from { opacity: 0; }\n  to { opacity: 1; }\n}\n\n@keyframes fadeOutBg {\n  from { opacity: 1; }\n  to { opacity: 0; }\n}\n\n@keyframes popOut {\n  from { opacity: 1; transform: translateY(0) scale(1); }\n  to { opacity: 0; transform: translateY(12px) scale(0.95); }\n}\n\n.flow-steps {\n  display: flex;\n  flex-direction: column;\n  gap: 8px;\n  margin-bottom: 14px;\n}\n\n.flow-step {\n  display: flex;\n  align-items: flex-start;\n  gap: 10px;\n  padding: 8px 10px;\n  background-color: var(--cream);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  font-size: 12px;\n}\n\n.flow-num {\n  width: 22px;\n  height: 22px;\n  border-radius: 50%;\n  background-color: var(--retro-yellow);\n  border: 2px solid var(--ink);\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  font-family: var(--font-mono);\n  font-weight: 900;\n  font-size: 12px;\n  flex-shrink: 0;\n}\n\n.win-patterns-box {\n  display: grid;\n  grid-template-columns: repeat(3, minmax(0, 1fr));\n  gap: 8px;\n  background-color: var(--retro-blue);\n  border: 2.5px solid var(--ink);\n  border-radius: var(--radius-sm);\n  padding: 10px;\n}\n\n.win-pattern-card {\n  background-color: var(--white);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  padding: 8px 6px;\n  display: flex;\n  flex-direction: column;\n  align-items: center;\n  justify-content: space-between;\n  gap: 6px;\n  font-size: 11px;\n  font-weight: 800;\n}\n\n.wp-grid {\n  display: grid;\n  grid-template-columns: repeat(4, 12px);\n  gap: 3px;\n  margin: auto 0;\n}\n\n.wp-grid.wp-vert { grid-template-columns: 12px; }\n.wp-grid.wp-diag { grid-template-columns: repeat(4, 10px); gap: 2px; }\n\n.wp-dot {\n  width: 100%;\n  aspect-ratio: 1 / 1;\n  border-radius: 50%;\n  border: 1.5px solid var(--ink);\n  background-color: var(--cream);\n}\n\n.wp-dot.coral { background-color: var(--retro-coral); }\n.wp-dot.yellow { background-color: var(--retro-yellow); }\n.wp-dot.green { background-color: var(--retro-green); }\n\n.theory-list {\n  display: flex;\n  flex-direction: column;\n  gap: 8px;\n  font-size: 12px;\n}\n\n.theory-item {\n  padding: 8px 10px;\n  border: 2px solid var(--ink);\n  border-left: 5px solid var(--retro-purple);\n  border-radius: var(--radius-sm);\n  background-color: var(--cream);\n}\n\n/* Entry Animation */\n.animate-pop-in {\n  animation: popIn 0.22s cubic-bezier(0.22, 1, 0.36, 1);\n}\n\n@keyframes popIn {\n  from { opacity: 0; transform: translateY(10px) scale(0.97); }\n  to { opacity: 1; transform: translateY(0) scale(1); }\n}\n\n/* Mobile Tight Fit (e.g. 400x741) */\n@media (max-width: 520px) {\n  .topbar-inner { padding: 8px 12px; }\n  .brand-title { font-size: 15px; }\n  .view-shell { padding: 14px 10px 10px; }\n  .retro-hero-head { padding: 18px 10px 12px; }\n  .retro-wordmark { font-size: 22px; }\n  .rw-num { font-size: 17px; }\n  .setup-head { padding: 15px 10px 8px; }\n  .setup-head-title { font-size: 11px; }\n  .setup-body { padding: 9px 10px 11px; }\n  .menu-section { margin-bottom: 8px; }\n  .choice-card { padding: 6px 7px; }\n  .choice-card strong { font-size: 11.5px; }\n  .choice-card span { font-size: 9.5px; }\n  .tile-badge { width: 20px; height: 20px; font-size: 11px; }\n  .arena-topbar { padding: 16px 8px 8px; }\n  .arena-body { padding: 9px 8px; }\n  .board-grid { padding: 6px; gap: 5px; }\n  .col-selectors, .hover-preview-row { gap: 5px; padding: 0 6px; }\n  .col-btn { font-size: 10px; padding: 4px 0; }\n}\n\n\nbody {\n  min-height: 760px !important;\n  padding-bottom: 24px !important;\n}\n.modal-backdrop {\n  align-items: flex-start;\n  padding-top: 32px;\n}\n\n</style>\n<!-- VIEW 1: MAIN MENU UTAMA (FIT 1 LAYAR) -->\n  <main class=\"view-shell view-centered\" id=\"homeView\">\n    <section class=\"home-stage animate-pop-in\">\n      <div class=\"card card-hero home-card\">\n        <div class=\"floating-pill\">MINIMAX AI</div>\n\n        <div class=\"retro-hero-head\">\n          <div class=\"retro-stripes\" aria-hidden=\"true\">\n            <span></span><span></span><span></span><span></span>\n          </div>\n          <div class=\"retro-wordmark\" aria-label=\"Connect 4\">\n            <span class=\"rw-1\">C</span>\n            <span class=\"rw-2\">O</span>\n            <span class=\"rw-3\">N</span>\n            <span class=\"rw-4\">N</span>\n            <span class=\"rw-5\">E</span>\n            <span class=\"rw-6\">C</span>\n            <span class=\"rw-7\">T</span>\n            <span class=\"rw-num\">4</span>\n          </div>\n          <div class=\"retro-stripes\" aria-hidden=\"true\">\n            <span></span><span></span><span></span><span></span>\n          </div>\n        </div>\n\n        <div class=\"hero-body home-body\">\n          <div class=\"showcase-wrap\" aria-hidden=\"true\">\n            <div class=\"showcase-board\">\n              <div class=\"sc-cell\"></div>\n              <div class=\"sc-cell\"></div>\n              <div class=\"sc-cell\"></div>\n              <div class=\"sc-cell sc-y sc-win\"></div>\n              <div class=\"sc-cell\"></div>\n\n              <div class=\"sc-cell\"></div>\n              <div class=\"sc-cell\"></div>\n              <div class=\"sc-cell sc-y sc-win\"></div>\n              <div class=\"sc-cell sc-c\"></div>\n              <div class=\"sc-cell\"></div>\n\n              <div class=\"sc-cell\"></div>\n              <div class=\"sc-cell sc-y sc-win\"></div>\n              <div class=\"sc-cell sc-c\"></div>\n              <div class=\"sc-cell sc-y\"></div>\n              <div class=\"sc-cell\"></div>\n\n              <div class=\"sc-cell sc-y sc-win\"></div>\n              <div class=\"sc-cell sc-c\"></div>\n              <div class=\"sc-cell sc-c\"></div>\n              <div class=\"sc-cell sc-c\"></div>\n              <div class=\"sc-cell sc-y\"></div>\n            </div>\n            <p class=\"showcase-tagline\">\n              Hubungkan 4 bidak sebelum Bot AI membaca langkahmu!\n            </p>\n          </div>\n\n          <div class=\"home-menu-buttons\">\n            <button type=\"button\" class=\"btn btn-primary btn-lg home-play-btn\" id=\"goSetupBtn\">\n              <svg class=\"btn-icon\" viewBox=\"0 0 24 24\" fill=\"currentColor\" aria-hidden=\"true\">\n                <polygon points=\"6,4 20,12 6,20\" />\n              </svg>\n              <span>MAIN</span>\n            </button>\n\n            <div class=\"home-sub-buttons\">\n              <button type=\"button\" class=\"btn btn-colab\" id=\"openRulesBtn\">\n                <svg class=\"btn-icon\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2.5\" aria-hidden=\"true\">\n                  <rect x=\"4\" y=\"4\" width=\"16\" height=\"16\" rx=\"2\" />\n                  <line x1=\"8\" y1=\"9\" x2=\"16\" y2=\"9\" />\n                  <line x1=\"8\" y1=\"13\" x2=\"16\" y2=\"13\" />\n                  <line x1=\"8\" y1=\"17\" x2=\"13\" y2=\"17\" />\n                </svg>\n                <span>Alur &amp; Aturan</span>\n              </button>\n              <button type=\"button\" class=\"btn btn-pink\" id=\"openInfoBtn\">\n                <svg class=\"btn-icon\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2.5\" aria-hidden=\"true\">\n                  <circle cx=\"12\" cy=\"5\" r=\"2.5\" />\n                  <circle cx=\"6\" cy=\"18\" r=\"2.5\" />\n                  <circle cx=\"18\" cy=\"18\" r=\"2.5\" />\n                  <line x1=\"11\" y1=\"7.5\" x2=\"7\" y2=\"15.5\" />\n                  <line x1=\"13\" y1=\"7.5\" x2=\"17\" y2=\"15.5\" />\n                </svg>\n                <span>Cara Kerja AI</span>\n              </button>\n            </div>\n          </div>\n        </div>\n      </div>\n    </section>\n  </main>\n\n  <!-- VIEW 2: MENU SETUP PERTANDINGAN (VISUAL & FIT 1 LAYAR) -->\n  <main class=\"view-shell view-centered\" id=\"setupView\" style=\"display: none;\">\n    <section class=\"setup-stage animate-pop-in\">\n      <div class=\"card card-hero setup-card\">\n        <div class=\"floating-pill\">SETUP PERTANDINGAN</div>\n\n        <div class=\"setup-head\">\n          <button type=\"button\" class=\"btn btn-sm\" id=\"backHomeBtn\">&larr; Menu</button>\n          <span class=\"setup-head-title\">Pilih Mode &amp; Level</span>\n          <span class=\"badge badge-accent\" id=\"setupPreviewBadge\">Hard • d=5</span>\n        </div>\n\n        <div class=\"hero-body setup-body\">\n          <div class=\"menu-section\">\n            <label class=\"menu-label\">1. Mode Permainan</label>\n            <div class=\"choice-grid cols-2-always\" id=\"modeChoiceGrid\">\n              <button type=\"button\" class=\"choice-card compact-row-card active\" data-mode=\"hvb\">\n                <div class=\"vc-icon-row\">\n                  <span class=\"mini-disc coral\"></span>\n                  <span class=\"vs-tag\">VS</span>\n                  <span class=\"bot-chip yellow\">AI</span>\n                </div>\n                <div>\n                  <strong>Manusia vs AI</strong>\n                  <span>Tantang Bot</span>\n                </div>\n              </button>\n              <button type=\"button\" class=\"choice-card compact-row-card\" data-mode=\"bvb\">\n                <div class=\"vc-icon-row\">\n                  <span class=\"bot-chip coral\">AI</span>\n                  <span class=\"vs-tag\">VS</span>\n                  <span class=\"bot-chip yellow\">AI</span>\n                </div>\n                <div>\n                  <strong>Bot vs Bot</strong>\n                  <span>Duel 2 agen AI</span>\n                </div>\n              </button>\n            </div>\n          </div>\n\n          <div class=\"menu-section\">\n            <div class=\"label-row-split\">\n              <label class=\"menu-label\" style=\"margin-bottom: 0;\">2. Mode Papan &amp; Tiles</label>\n              <div class=\"shape-toggle\" id=\"tileShapeGrid\" role=\"group\" aria-label=\"Bentuk Tiles\">\n                <button type=\"button\" class=\"shape-btn active\" data-shape=\"circle\" title=\"Bidak Bulat\">\n                  <span class=\"shape-ico circle\"></span>Bulat\n                </button>\n                <button type=\"button\" class=\"shape-btn\" data-shape=\"square\" title=\"Tile Kotak Retro\">\n                  <span class=\"shape-ico square\"></span>Kotak\n                </button>\n              </div>\n            </div>\n            <div class=\"choice-grid cols-3-always\" id=\"boardModeGrid\">\n              <button type=\"button\" class=\"choice-card tile-mode-card active\" data-bmode=\"classic\">\n                <span class=\"tile-badge blue\">4×</span>\n                <strong>Klasik</strong>\n                <span>Papan standar</span>\n              </button>\n              <button type=\"button\" class=\"choice-card tile-mode-card\" data-bmode=\"obstacle\">\n                <span class=\"tile-badge stone\">&#x2715;</span>\n                <strong>Obstacle</strong>\n                <span>Batu rintangan</span>\n              </button>\n              <button type=\"button\" class=\"choice-card tile-mode-card\" data-bmode=\"star\">\n                <span class=\"tile-badge star\">&#9733;</span>\n                <strong>Star +6</strong>\n                <span>Petak bonus</span>\n              </button>\n            </div>\n\n            <div class=\"board-size-bar\">\n              <span class=\"size-label\">Ukuran Papan:</span>\n              <div class=\"size-toggle\" id=\"boardSizeGrid\" role=\"group\" aria-label=\"Ukuran Papan\">\n                <button type=\"button\" class=\"size-btn active\" data-size=\"6x7\">6×7 Standar</button>\n                <button type=\"button\" class=\"size-btn\" data-size=\"7x8\">7×8 Luas</button>\n                <button type=\"button\" class=\"size-btn\" data-size=\"7x9\">7×9 Ekstra</button>\n              </div>\n            </div>\n          </div>\n\n          <div class=\"menu-section\">\n            <label class=\"menu-label\">3. Tingkat Kesulitan</label>\n            <div class=\"choice-grid cols-2-always\" id=\"diffChoiceGrid\">\n              <button type=\"button\" class=\"choice-card diff-card\" data-diff=\"Easy\">\n                <div class=\"diff-top\">\n                  <strong>Easy</strong>\n                  <span class=\"power-bars\" aria-hidden=\"true\">\n                    <i class=\"bar on green\"></i><i class=\"bar\"></i><i class=\"bar\"></i>\n                  </span>\n                </div>\n                <span>Langkah acak</span>\n              </button>\n              <button type=\"button\" class=\"choice-card diff-card\" data-diff=\"Medium\">\n                <div class=\"diff-top\">\n                  <strong>Medium</strong>\n                  <span class=\"power-bars\" aria-hidden=\"true\">\n                    <i class=\"bar on yellow\"></i><i class=\"bar on yellow\"></i><i class=\"bar\"></i>\n                  </span>\n                </div>\n                <span>Minimax Depth 3</span>\n              </button>\n              <button type=\"button\" class=\"choice-card diff-card active\" data-diff=\"Hard\">\n                <div class=\"diff-top\">\n                  <strong>Hard</strong>\n                  <span class=\"power-bars\" aria-hidden=\"true\">\n                    <i class=\"bar on coral\"></i><i class=\"bar on coral\"></i><i class=\"bar on coral\"></i>\n                  </span>\n                </div>\n                <span>Alpha-Beta Depth 5</span>\n              </button>\n              <button type=\"button\" class=\"choice-card diff-card\" data-diff=\"Custom\">\n                <div class=\"diff-top\">\n                  <strong>Custom</strong>\n                  <span class=\"custom-chip\">SET</span>\n                </div>\n                <span>Atur algo &amp; depth</span>\n              </button>\n            </div>\n          </div>\n\n          <div class=\"menu-section custom-box\" id=\"customConfigBox\" style=\"display: none;\">\n            <div class=\"controls-grid cols-2-always\">\n              <div class=\"control-group\">\n                <label for=\"algorithmSelect\">Algoritma</label>\n                <select id=\"algorithmSelect\">\n                  <option value=\"alphabeta\" selected>Alpha-Beta</option>\n                  <option value=\"minimax\">Minimax Murni</option>\n                </select>\n              </div>\n              <div class=\"control-group\">\n                <label for=\"depthSelect\">Max Depth</label>\n                <select id=\"depthSelect\">\n                  <option value=\"1\">Depth 1</option>\n                  <option value=\"2\">Depth 2</option>\n                  <option value=\"3\">Depth 3</option>\n                  <option value=\"4\">Depth 4</option>\n                  <option value=\"5\" selected>Depth 5</option>\n                </select>\n              </div>\n            </div>\n          </div>\n\n          <div class=\"menu-section\">\n            <label class=\"menu-label\">4. Jalan Pertama</label>\n            <div class=\"choice-grid cols-3-always\" id=\"firstChoiceGrid\">\n              <button type=\"button\" class=\"choice-card first-card active\" data-first=\"1\">\n                <span class=\"first-disc coral\"></span>\n                <div>\n                  <strong id=\"firstP1Title\">Manusia</strong>\n                  <span>Coral</span>\n                </div>\n              </button>\n              <button type=\"button\" class=\"choice-card first-card\" data-first=\"2\">\n                <span class=\"first-disc yellow\"></span>\n                <div>\n                  <strong>Bot AI</strong>\n                  <span>Kuning</span>\n                </div>\n              </button>\n              <button type=\"button\" class=\"choice-card first-card\" data-first=\"random\">\n                <span class=\"first-disc random\">?</span>\n                <div>\n                  <strong>Acak</strong>\n                  <span>Undi 50:50</span>\n                </div>\n              </button>\n            </div>\n          </div>\n\n          <div class=\"setup-actions\">\n            <button type=\"button\" class=\"btn btn-primary btn-lg\" id=\"startGameBtn\">\n              <svg class=\"btn-icon\" viewBox=\"0 0 24 24\" fill=\"currentColor\" aria-hidden=\"true\">\n                <polygon points=\"6,4 20,12 6,20\" />\n              </svg>\n              <span>MULAI PERTANDINGAN</span>\n            </button>\n          </div>\n        </div>\n      </div>\n    </section>\n  </main>\n\n  <!-- VIEW 3 & 4: ARENA BERSIH -> MUNCUL ANALISIS SETELAH SELESAI -->\n  <main class=\"view-shell\" id=\"gameView\" style=\"display: none;\">\n    <div class=\"arena-shell\" id=\"arenaShell\">\n      <section class=\"arena-column\">\n        <div class=\"card card-hero\">\n          <div class=\"floating-pill\" id=\"arenaPill\">ARENA MAIN</div>\n\n          <div class=\"arena-topbar\">\n            <div class=\"arena-nav-btns\">\n              <button type=\"button\" class=\"btn btn-sm\" id=\"backMenuBtn\">&larr; Menu</button>\n              <button type=\"button\" class=\"btn btn-sm btn-colab\" id=\"backSetupBtn\">Setup</button>\n            </div>\n            <div class=\"arena-badges\">\n              <span class=\"badge badge-pink\" id=\"activeConfigBadge\">Hard • d=5</span>\n              <span class=\"badge badge-accent\" id=\"turnCounterBadge\">#0</span>\n            </div>\n            <div class=\"arena-quick-btns\">\n              <button type=\"button\" class=\"btn btn-sm btn-pink\" id=\"undoBtn\">Undo</button>\n              <button type=\"button\" class=\"btn btn-sm btn-primary\" id=\"resetBtn\">Reset</button>\n            </div>\n          </div>\n\n          <div class=\"arena-bvb-bar\" id=\"bvbBar\" style=\"display: none;\">\n            <button type=\"button\" class=\"btn btn-sm btn-colab\" id=\"botStepBtn\">1 Langkah</button>\n            <button type=\"button\" class=\"btn btn-sm btn-green\" id=\"autoPlayBtn\">Jeda Auto-Play</button>\n          </div>\n\n          <div class=\"hero-body arena-body\">\n            <div class=\"status-banner turn-p1\" id=\"statusBanner\" role=\"status\" aria-live=\"polite\">\n              <div class=\"status-left\">\n                <span class=\"status-dot p1\" id=\"statusDot\"></span>\n                <span id=\"statusText\">Giliran Anda — Pilih Kolom 0–6</span>\n                <span class=\"thinking-dots\" id=\"thinkingDots\" aria-hidden=\"true\">\n                  <span></span><span></span><span></span>\n                </span>\n              </div>\n              <span class=\"badge\" id=\"lastMovePill\" style=\"display: none;\"></span>\n            </div>\n\n            <div class=\"board-wrapper\">\n              <div class=\"hover-preview-row\" id=\"hoverPreviewRow\" aria-hidden=\"true\"></div>\n              <div class=\"col-selectors\" id=\"colSelectors\" aria-label=\"Pilih kolom\"></div>\n              <div class=\"board-grid\" id=\"boardGrid\" role=\"grid\" aria-label=\"Papan Connect-Four 6 baris 7 kolom\"></div>\n              <div class=\"board-legend\">\n                <span class=\"legend-item\">\n                  <span class=\"legend-swatch p1\" id=\"p1Swatch\"></span>\n                  <span id=\"p1LegendLabel\">Manusia</span>\n                </span>\n                <span class=\"legend-item\">\n                  <span class=\"legend-swatch p2\" id=\"p2Swatch\"></span>\n                  <span id=\"p2LegendLabel\">Bot AI</span>\n                </span>\n                <span class=\"legend-item\" id=\"specialTileLegend\" style=\"display: none;\"></span>\n              </div>\n            </div>\n          </div>\n        </div>\n      </section>\n\n      <section class=\"analysis-column\" id=\"postGamePanel\" style=\"display: none;\">\n        <div class=\"card animate-pop-in\">\n          <div class=\"floating-pill\">HASIL &amp; ANALISIS</div>\n\n          <div class=\"postgame-banner\" id=\"postGameBanner\">\n            <div>\n              <div class=\"postgame-title\" id=\"postGameTitle\">Permainan Selesai!</div>\n              <div class=\"postgame-sub\" id=\"postGameSub\">Ringkasan performa Minimax selama pertandingan.</div>\n            </div>\n            <div class=\"postgame-cta\">\n              <button type=\"button\" class=\"btn btn-sm btn-colab\" id=\"playAgainBtn\">Main Lagi</button>\n              <button type=\"button\" class=\"btn btn-sm\" id=\"postMenuBtn\">Menu</button>\n            </div>\n          </div>\n\n          <div class=\"metrics-grid\">\n            <div class=\"metric-box accent-yellow\">\n              <div class=\"metric-label\">Total Langkah</div>\n              <div class=\"metric-value\" id=\"statTotalMoves\">0</div>\n            </div>\n            <div class=\"metric-box\">\n              <div class=\"metric-label\">Langkah Bot</div>\n              <div class=\"metric-value\" id=\"statBotMoves\">0</div>\n            </div>\n            <div class=\"metric-box accent-pink\">\n              <div class=\"metric-label\">Rata-rata Node</div>\n              <div class=\"metric-value\" id=\"statAvgNodes\">0</div>\n            </div>\n            <div class=\"metric-box\">\n              <div class=\"metric-label\">Total Waktu Bot</div>\n              <div class=\"metric-value\" id=\"statTotalTime\">0 ms</div>\n            </div>\n          </div>\n\n          <div class=\"section-label\">Tabel Analisis Langkah Bot</div>\n          <div class=\"analysis-table-wrap\">\n            <table class=\"analysis-table\">\n              <thead>\n                <tr>\n                  <th>#</th>\n                  <th>Aktor</th>\n                  <th>Kolom</th>\n                  <th>Skor Eval</th>\n                  <th>Node</th>\n                  <th>Waktu</th>\n                </tr>\n              </thead>\n              <tbody id=\"postGameTableBody\"></tbody>\n            </table>\n          </div>\n\n          <div class=\"scoring-legend-card\" style=\"margin-top: 14px;\">\n            <div class=\"scoring-legend-head\">\n              <span>Scoring Legend — Papan Akhir</span>\n              <span class=\"badge\" id=\"legendTotalBadge\">Total: 0</span>\n            </div>\n            <div class=\"scoring-legend-list\" id=\"heuristicGrid\"></div>\n          </div>\n        </div>\n\n        <div class=\"card animate-pop-in\" style=\"margin-top: 18px;\">\n          <div class=\"card-header\">\n            <h2 class=\"card-title\">Uji Efisiensi: Minimax vs Alpha-Beta</h2>\n            <div style=\"display: flex; gap: 6px; align-items: center;\">\n              <select id=\"compareDepthSelect\" class=\"mini-select\" aria-label=\"Pilih kedalaman benchmark\">\n                <option value=\"3\">Depth 3</option>\n                <option value=\"4\" selected>Depth 4</option>\n                <option value=\"5\">Depth 5</option>\n              </select>\n              <button class=\"btn btn-green btn-sm\" id=\"compareBtn\" type=\"button\">Bandingkan Node</button>\n            </div>\n          </div>\n          <p class=\"compare-caption\" id=\"compareContextText\">\n            Bandingkan jumlah node pada posisi papan aktif terakhir sebelum langkah penutup.\n          </p>\n          <div class=\"compare-result\" id=\"compareResult\">\n            Klik <strong>Bandingkan Node</strong> untuk melihat selisih efisiensi Minimax vs Alpha-Beta.\n          </div>\n        </div>\n      </section>\n    </div>\n  </main>\n\n  <!-- MODAL ALUR & ATURAN MAIN -->\n  <div class=\"modal-backdrop\" id=\"rulesModal\" style=\"display: none;\" role=\"dialog\" aria-modal=\"true\" aria-labelledby=\"rulesModalTitle\">\n    <div class=\"card modal-card\">\n      <div class=\"card-header\">\n        <h2 class=\"card-title\" id=\"rulesModalTitle\">Alur &amp; Mode Permainan</h2>\n        <button type=\"button\" class=\"btn btn-sm btn-coral\" id=\"closeRulesBtn\">Tutup</button>\n      </div>\n\n      <div class=\"flow-steps\">\n        <div class=\"flow-step\">\n          <span class=\"flow-num\">1</span>\n          <div><strong>Setup:</strong> Pilih Mode, Tipe Papan (<em>Klasik</em>, <em>Obstacle</em>, <em>Star</em>), Ukuran (<em>6×7</em>, <em>7×8</em>, <em>7×9</em>), Bentuk Tiles, Level AI, &amp; giliran pertama (bisa <strong>Acak</strong>).</div>\n        </div>\n        <div class=\"flow-step\">\n          <span class=\"flow-num\">2</span>\n          <div><strong>Jatuhkan Bidak:</strong> Klik kolom <code>K0–K8</code> sesuai ukuran papan (atau keyboard angka), bidak jatuh ke petak kosong terbawah.</div>\n        </div>\n        <div class=\"flow-step\">\n          <span class=\"flow-num\">3</span>\n          <div><strong>Mode Spesial:</strong> Di <em>Obstacle</em> ada batu yang memblokir jalur 4-in-a-row; di <em>Star</em> ada petak <code>&#9733;</code> bernilai bonus skor evaluasi <code>+6</code>.</div>\n        </div>\n        <div class=\"flow-step\">\n          <span class=\"flow-num\">4</span>\n          <div><strong>Analisis AI:</strong> Setelah permainan selesai, tabel skor evaluasi &amp; perbandingan Minimax vs Alpha-Beta langsung muncul.</div>\n        </div>\n      </div>\n\n      <div class=\"win-patterns-box\">\n        <div class=\"win-pattern-card\">\n          <div class=\"wp-grid\">\n            <span class=\"wp-dot coral\"></span><span class=\"wp-dot coral\"></span><span class=\"wp-dot coral\"></span><span class=\"wp-dot coral\"></span>\n          </div>\n          <span>Horizontal</span>\n        </div>\n        <div class=\"win-pattern-card\">\n          <div class=\"wp-grid wp-vert\">\n            <span class=\"wp-dot yellow\"></span><span class=\"wp-dot yellow\"></span><span class=\"wp-dot yellow\"></span><span class=\"wp-dot yellow\"></span>\n          </div>\n          <span>Vertikal</span>\n        </div>\n        <div class=\"win-pattern-card\">\n          <div class=\"wp-grid wp-diag\">\n            <span class=\"wp-dot green\"></span><span class=\"wp-dot\"></span><span class=\"wp-dot\"></span><span class=\"wp-dot\"></span>\n            <span class=\"wp-dot\"></span><span class=\"wp-dot green\"></span><span class=\"wp-dot\"></span><span class=\"wp-dot\"></span>\n            <span class=\"wp-dot\"></span><span class=\"wp-dot\"></span><span class=\"wp-dot green\"></span><span class=\"wp-dot\"></span>\n            <span class=\"wp-dot\"></span><span class=\"wp-dot\"></span><span class=\"wp-dot\"></span><span class=\"wp-dot green\"></span>\n          </div>\n          <span>Diagonal</span>\n        </div>\n      </div>\n    </div>\n  </div>\n\n  <!-- MODAL INFO BOBOT EVAL & CARA KERJA AI -->\n  <div class=\"modal-backdrop\" id=\"infoModal\" style=\"display: none;\" role=\"dialog\" aria-modal=\"true\" aria-labelledby=\"infoModalTitle\">\n    <div class=\"card modal-card\">\n      <div class=\"card-header\">\n        <h2 class=\"card-title\" id=\"infoModalTitle\">Cara Kerja AI &amp; Bobot Skor</h2>\n        <button type=\"button\" class=\"btn btn-sm btn-coral\" id=\"closeInfoBtn\">Tutup</button>\n      </div>\n      <div class=\"theory-list\" style=\"margin-bottom: 12px;\">\n        <div class=\"theory-item\">\n          <strong>Depth-Limited Minimax:</strong> Pencarian Game Tree dibatasi hingga <code>max_depth</code> dan dinilai memakai fungsi evaluasi papan.\n        </div>\n        <div class=\"theory-item\">\n          <strong>Alpha-Beta Pruning:</strong> Memangkas cabang yang tidak memengaruhi hasil saat <code>&alpha; &ge; &beta;</code> dengan prioritas kolom tengah <code>[3, 2, 4, 1, 5, 0, 6]</code>.\n        </div>\n      </div>\n      <div class=\"scoring-legend-card\">\n        <div class=\"scoring-legend-head\">\n          <span>Scoring Legend — Bobot Eval</span>\n        </div>\n        <div class=\"scoring-legend-list\" id=\"modalLegendList\"></div>\n      </div>\n    </div>\n  </div>\n<script>\n/**\n * Connect-Four AI Engine — TIM 12 UTS PjBL Kecerdasan Buatan\n * Sinkron 1:1 dengan implementasi Python di Praktikum_AI_UTS_TIM12.ipynb.\n *\n * Fitur:\n * - Adversarial Search: Depth-Limited Minimax & Alpha-Beta Pruning\n * - Heuristic Board Evaluation Function (4-cell window + kontrol tengah + bonus Star Tiles)\n * - Dukungan variasi papan & ukuran dinamis: 6x7, 7x8, 7x9 + Classic, Obstacle, Star\n */\n(function (root, factory) {\n  if (typeof module === 'object' && module.exports) {\n    module.exports = factory();\n  } else {\n    root.ConnectFourAI = factory();\n  }\n})(typeof globalThis !== 'undefined' ? globalThis : this, function () {\n  'use strict';\n\n  const ROWS = 6;\n  const COLS = 7;\n  const WINDOW_LENGTH = 4;\n\n  const BLOCKED = -1;\n  const EMPTY = 0;\n  const HUMAN = 1;\n  const AI = 2;\n\n  const WIN_SCORE = 100000;\n  const COLUMN_ORDER = [3, 2, 4, 1, 5, 0, 6];\n\n  const BOARD_SIZES = {\n    '6x7': { rows: 6, cols: 7, label: '6×7' },\n    '7x8': { rows: 7, cols: 8, label: '7×8' },\n    '7x9': { rows: 7, cols: 9, label: '7×9' },\n  };\n\n  const EVAL_WEIGHTS = {\n    WIN: 100000,\n    AI_THREE: 12,\n    AI_TWO: 3,\n    OPP_THREE: -80,\n    OPP_TWO: -2,\n    CENTER_PER_DISC: 4,\n    STAR_BONUS: 6,\n  };\n\n  const DIFFICULTY_PRESETS = {\n    Easy: { maxDepth: 0, usePruning: false, label: 'Easy (Random Move)' },\n    Medium: { maxDepth: 3, usePruning: false, label: 'Medium (Depth-Limited Minimax, depth=3)' },\n    Hard: { maxDepth: 5, usePruning: true, label: 'Hard (Depth-Limited + Alpha-Beta, depth=5)' },\n  };\n\n  /**\n   * Kembalikan urutan kolom tengah-ke-pinggir untuk jumlah kolom berapa pun.\n   */\n  function getColumnOrder(cols = COLS) {\n    const order = [];\n    if (cols % 2 === 1) {\n      const mid = Math.floor(cols / 2);\n      order.push(mid);\n      for (let d = 1; d <= mid; d++) {\n        order.push(mid - d);\n        order.push(mid + d);\n      }\n    } else {\n      const left = cols / 2 - 1;\n      const right = cols / 2;\n      for (let d = 0; d < cols / 2; d++) {\n        order.push(left - d);\n        order.push(right + d);\n      }\n    }\n    return order;\n  }\n\n  /**\n   * Buat papan kosong berukuran rows x cols (baris 0 = paling atas).\n   */\n  function createBoard(rows = ROWS, cols = COLS) {\n    const board = [];\n    for (let r = 0; r < rows; r++) {\n      board.push(new Array(cols).fill(EMPTY));\n    }\n    return board;\n  }\n\n  /**\n   * Buat konfigurasi awal papan sesuai mode ('classic', 'obstacle', 'star') dan ukuran rows x cols.\n   * Mengembalikan { board, starCells, obstacles }.\n   */\n  function createSpecialSetup(boardMode = 'classic', rows = ROWS, cols = COLS) {\n    const board = createBoard(rows, cols);\n    const starCells = [];\n    const obstacles = [];\n\n    if (boardMode === 'obstacle') {\n      const centerSet = new Set(\n        cols % 2 === 1 ? [Math.floor(cols / 2)] : [cols / 2 - 1, cols / 2]\n      );\n      const candidateCols = [];\n      for (let c = 0; c < cols; c++) {\n        if (!centerSet.has(c)) candidateCols.push(c);\n      }\n      for (let i = candidateCols.length - 1; i > 0; i--) {\n        const j = Math.floor(Math.random() * (i + 1));\n        const tmp = candidateCols[i];\n        candidateCols[i] = candidateCols[j];\n        candidateCols[j] = tmp;\n      }\n\n      const spots = [\n        [rows - 1, candidateCols[0]],\n        [rows - 1, candidateCols[1]],\n        [rows - 2, candidateCols[1]],\n        [rows - 1, candidateCols[2]],\n      ];\n      if (cols >= 8 && candidateCols.length >= 4) {\n        spots.push([rows - 1, candidateCols[3]]);\n      }\n\n      spots.forEach(([r, c]) => {\n        board[r][c] = BLOCKED;\n        obstacles.push([r, c]);\n      });\n    } else if (boardMode === 'star') {\n      const colPool = [];\n      for (let c = 1; c < cols - 1; c++) {\n        colPool.push(c);\n      }\n      for (let i = colPool.length - 1; i > 0; i--) {\n        const j = Math.floor(Math.random() * (i + 1));\n        const tmp = colPool[i];\n        colPool[i] = colPool[j];\n        colPool[j] = tmp;\n      }\n      const starCount = cols >= 8 ? 4 : 3;\n      const minRow = Math.max(2, rows - 4);\n      for (let i = 0; i < Math.min(starCount, colPool.length); i++) {\n        const r = minRow + Math.floor(Math.random() * (rows - minRow));\n        starCells.push([r, colPool[i]]);\n      }\n    }\n\n    return { board, starCells, obstacles };\n  }\n\n  /**\n   * Salin papan 2D secara mendalam.\n   */\n  function cloneBoard(board) {\n    return board.map((row) => row.slice());\n  }\n\n  /**\n   * Cek apakah kolom masih memiliki ruang kosong di paling atas.\n   */\n  function isValidLocation(board, col) {\n    const cols = board[0].length;\n    return col >= 0 && col < cols && board[0][col] === EMPTY;\n  }\n\n  /**\n   * Kembalikan daftar kolom valid sesuai urutan tengah-ke-pinggir.\n   */\n  function getValidLocations(board) {\n    const order = getColumnOrder(board[0].length);\n    const valid = [];\n    for (let i = 0; i < order.length; i++) {\n      const col = order[i];\n      if (isValidLocation(board, col)) {\n        valid.push(col);\n      }\n    }\n    return valid;\n  }\n\n  /**\n   * Cari baris terbawah yang masih kosong pada kolom tertentu.\n   */\n  function getNextOpenRow(board, col) {\n    for (let r = board.length - 1; r >= 0; r--) {\n      if (board[r][col] === EMPTY) {\n        return r;\n      }\n    }\n    return -1;\n  }\n\n  /**\n   * Jatuhkan bidak ke koordinat (row, col).\n   */\n  function dropPiece(board, row, col, piece) {\n    board[row][col] = piece;\n  }\n\n  /**\n   * Cek pemenang atau seri beserta koordinat 4 bidak yang terhubung.\n   * Mengembalikan { winner: 1 | 2 | 'DRAW' | null, cells: [[r, c], ...] }.\n   */\n  function getWinnerInfo(board) {\n    const rows = board.length;\n    const cols = board[0].length;\n    const isPlayer = (p) => p === HUMAN || p === AI;\n\n    // Cek horizontal\n    for (let r = 0; r < rows; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        const p = board[r][c];\n        if (isPlayer(p) && p === board[r][c + 1] && p === board[r][c + 2] && p === board[r][c + 3]) {\n          return { winner: p, cells: [[r, c], [r, c + 1], [r, c + 2], [r, c + 3]] };\n        }\n      }\n    }\n\n    // Cek vertikal\n    for (let c = 0; c < cols; c++) {\n      for (let r = 0; r <= rows - WINDOW_LENGTH; r++) {\n        const p = board[r][c];\n        if (isPlayer(p) && p === board[r + 1][c] && p === board[r + 2][c] && p === board[r + 3][c]) {\n          return { winner: p, cells: [[r, c], [r + 1, c], [r + 2, c], [r + 3, c]] };\n        }\n      }\n    }\n\n    // Cek diagonal turun ke kanan (\\)\n    for (let r = 0; r <= rows - WINDOW_LENGTH; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        const p = board[r][c];\n        if (isPlayer(p) && p === board[r + 1][c + 1] && p === board[r + 2][c + 2] && p === board[r + 3][c + 3]) {\n          return { winner: p, cells: [[r, c], [r + 1, c + 1], [r + 2, c + 2], [r + 3, c + 3]] };\n        }\n      }\n    }\n\n    // Cek diagonal naik ke kanan (/)\n    for (let r = WINDOW_LENGTH - 1; r < rows; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        const p = board[r][c];\n        if (isPlayer(p) && p === board[r - 1][c + 1] && p === board[r - 2][c + 2] && p === board[r - 3][c + 3]) {\n          return { winner: p, cells: [[r, c], [r - 1, c + 1], [r - 2, c + 2], [r - 3, c + 3]] };\n        }\n      }\n    }\n\n    // Cek seri jika semua kolom teratas penuh\n    if (getValidLocations(board).length === 0) {\n      return { winner: 'DRAW', cells: [] };\n    }\n\n    return { winner: null, cells: [] };\n  }\n\n  /**\n   * Wrapper sederhana mengembalikan 1, 2, 'DRAW', atau null.\n   */\n  function checkWinner(board) {\n    return getWinnerInfo(board).winner;\n  }\n\n  /**\n   * Hitung skor satu jendela berisi 4 petak dari sudut pandang aiPiece.\n   */\n  function evaluateWindow(windowCells, aiPiece) {\n    const oppPiece = aiPiece === HUMAN ? AI : HUMAN;\n    let aiCount = 0;\n    let oppCount = 0;\n    let emptyCount = 0;\n\n    for (let i = 0; i < windowCells.length; i++) {\n      const cell = windowCells[i];\n      if (cell === BLOCKED) return 0;\n      if (cell === aiPiece) aiCount++;\n      else if (cell === oppPiece) oppCount++;\n      else emptyCount++;\n    }\n\n    if (aiCount === 4) return EVAL_WEIGHTS.WIN;\n    if (oppCount === 4) return -EVAL_WEIGHTS.WIN;\n    if (aiCount === 3 && emptyCount === 1) return EVAL_WEIGHTS.AI_THREE;\n    if (aiCount === 2 && emptyCount === 2) return EVAL_WEIGHTS.AI_TWO;\n    if (oppCount === 3 && emptyCount === 1) return EVAL_WEIGHTS.OPP_THREE;\n    if (oppCount === 2 && emptyCount === 2) return EVAL_WEIGHTS.OPP_TWO;\n    return 0;\n  }\n\n  /**\n   * Analisis papan menyeluruh untuk ukuran rows x cols berapa pun.\n   */\n  function analyzeBoard(board, aiPiece = AI, starCells = []) {\n    const rows = board.length;\n    const cols = board[0].length;\n    const oppPiece = aiPiece === HUMAN ? AI : HUMAN;\n\n    const centerCols = cols % 2 === 1 ? [Math.floor(cols / 2)] : [cols / 2 - 1, cols / 2];\n    const centerWeight = cols % 2 === 1 ? EVAL_WEIGHTS.CENTER_PER_DISC : Math.floor(EVAL_WEIGHTS.CENTER_PER_DISC / 2);\n\n    let centerAi = 0;\n    let centerOpp = 0;\n    for (let i = 0; i < centerCols.length; i++) {\n      const cc = centerCols[i];\n      for (let r = 0; r < rows; r++) {\n        if (board[r][cc] === aiPiece) centerAi++;\n        else if (board[r][cc] === oppPiece) centerOpp++;\n      }\n    }\n\n    let aiThree = 0;\n    let aiTwo = 0;\n    let oppThree = 0;\n    let oppTwo = 0;\n    let windowScore = 0;\n\n    function processWindow(w) {\n      let aiCount = 0;\n      let oppCount = 0;\n      let emptyCount = 0;\n      for (let i = 0; i < 4; i++) {\n        if (w[i] === BLOCKED) return;\n        if (w[i] === aiPiece) aiCount++;\n        else if (w[i] === oppPiece) oppCount++;\n        else emptyCount++;\n      }\n      if (aiCount === 3 && emptyCount === 1) aiThree++;\n      else if (aiCount === 2 && emptyCount === 2) aiTwo++;\n      else if (oppCount === 3 && emptyCount === 1) oppThree++;\n      else if (oppCount === 2 && emptyCount === 2) oppTwo++;\n      windowScore += evaluateWindow(w, aiPiece);\n    }\n\n    // Horizontal\n    for (let r = 0; r < rows; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        processWindow([board[r][c], board[r][c + 1], board[r][c + 2], board[r][c + 3]]);\n      }\n    }\n\n    // Vertikal\n    for (let c = 0; c < cols; c++) {\n      for (let r = 0; r <= rows - WINDOW_LENGTH; r++) {\n        processWindow([board[r][c], board[r + 1][c], board[r + 2][c], board[r + 3][c]]);\n      }\n    }\n\n    // Diagonal turun (\\)\n    for (let r = 0; r <= rows - WINDOW_LENGTH; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        processWindow([board[r][c], board[r + 1][c + 1], board[r + 2][c + 2], board[r + 3][c + 3]]);\n      }\n    }\n\n    // Diagonal naik (/)\n    for (let r = WINDOW_LENGTH - 1; r < rows; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        processWindow([board[r][c], board[r - 1][c + 1], board[r - 2][c + 2], board[r - 3][c + 3]]);\n      }\n    }\n\n    let starAi = 0;\n    let starOpp = 0;\n    if (Array.isArray(starCells) && starCells.length > 0) {\n      for (let i = 0; i < starCells.length; i++) {\n        const [sr, sc] = starCells[i];\n        if (sr >= 0 && sr < rows && sc >= 0 && sc < cols) {\n          if (board[sr][sc] === aiPiece) starAi++;\n          else if (board[sr][sc] === oppPiece) starOpp++;\n        }\n      }\n    }\n\n    const centerScore = (centerAi - centerOpp) * centerWeight;\n    const starScore = (starAi - starOpp) * EVAL_WEIGHTS.STAR_BONUS;\n    return {\n      totalScore: windowScore + centerScore + starScore,\n      centerAi,\n      centerOpp,\n      centerScore,\n      starAi,\n      starOpp,\n      starScore,\n      aiThree,\n      aiTwo,\n      oppThree,\n      oppTwo,\n      windowScore,\n    };\n  }\n\n  /**\n   * Fungsi Evaluasi Papan (Evaluation Function) untuk non-terminal state.\n   */\n  function evaluateBoard(board, aiPiece = AI, starCells = []) {\n    return analyzeBoard(board, aiPiece, starCells).totalScore;\n  }\n\n  /**\n   * Depth-Limited Minimax Murni (tanpa Alpha-Beta Pruning).\n   */\n  function minimax(board, depth, maxDepth, isMaximizing, aiPiece, stats, starCells = []) {\n    stats.nodes += 1;\n    const oppPiece = aiPiece === HUMAN ? AI : HUMAN;\n\n    const winner = checkWinner(board);\n    if (winner === aiPiece) return WIN_SCORE - depth;\n    if (winner === oppPiece) return depth - WIN_SCORE;\n    if (winner === 'DRAW') return 0;\n\n    if (depth >= maxDepth) {\n      return evaluateBoard(board, aiPiece, starCells);\n    }\n\n    const validCols = getValidLocations(board);\n\n    if (isMaximizing) {\n      let bestScore = -Infinity;\n      for (let i = 0; i < validCols.length; i++) {\n        const col = validCols[i];\n        const row = getNextOpenRow(board, col);\n        board[row][col] = aiPiece;\n        const score = minimax(board, depth + 1, maxDepth, false, aiPiece, stats, starCells);\n        board[row][col] = EMPTY;\n        if (score > bestScore) bestScore = score;\n      }\n      return bestScore;\n    }\n\n    let bestScore = Infinity;\n    for (let i = 0; i < validCols.length; i++) {\n      const col = validCols[i];\n      const row = getNextOpenRow(board, col);\n      board[row][col] = oppPiece;\n      const score = minimax(board, depth + 1, maxDepth, true, aiPiece, stats, starCells);\n      board[row][col] = EMPTY;\n      if (score < bestScore) bestScore = score;\n    }\n    return bestScore;\n  }\n\n  /**\n   * Depth-Limited Minimax dengan optimasi Alpha-Beta Pruning.\n   */\n  function minimaxAlphaBeta(board, depth, maxDepth, alpha, beta, isMaximizing, aiPiece, stats, starCells = []) {\n    stats.nodes += 1;\n    const oppPiece = aiPiece === HUMAN ? AI : HUMAN;\n\n    const winner = checkWinner(board);\n    if (winner === aiPiece) return WIN_SCORE - depth;\n    if (winner === oppPiece) return depth - WIN_SCORE;\n    if (winner === 'DRAW') return 0;\n\n    if (depth >= maxDepth) {\n      return evaluateBoard(board, aiPiece, starCells);\n    }\n\n    const validCols = getValidLocations(board);\n\n    if (isMaximizing) {\n      let maxEval = -Infinity;\n      for (let i = 0; i < validCols.length; i++) {\n        const col = validCols[i];\n        const row = getNextOpenRow(board, col);\n        board[row][col] = aiPiece;\n        const evalScore = minimaxAlphaBeta(board, depth + 1, maxDepth, alpha, beta, false, aiPiece, stats, starCells);\n        board[row][col] = EMPTY;\n        if (evalScore > maxEval) maxEval = evalScore;\n        if (maxEval > alpha) alpha = maxEval;\n        if (alpha >= beta) break;\n      }\n      return maxEval;\n    }\n\n    let minEval = Infinity;\n    for (let i = 0; i < validCols.length; i++) {\n      const col = validCols[i];\n      const row = getNextOpenRow(board, col);\n      board[row][col] = oppPiece;\n      const evalScore = minimaxAlphaBeta(board, depth + 1, maxDepth, alpha, beta, true, aiPiece, stats, starCells);\n      board[row][col] = EMPTY;\n      if (evalScore < minEval) minEval = evalScore;\n      if (minEval < beta) beta = minEval;\n      if (alpha >= beta) break;\n    }\n    return minEval;\n  }\n\n  /**\n   * Cari langkah terbaik beserta skor evaluasi tiap kolom dan metrik pencarian.\n   */\n  function findBestMove(board, options = {}) {\n    const maxDepth = options.maxDepth !== undefined ? options.maxDepth : 4;\n    const aiPiece = options.aiPiece !== undefined ? options.aiPiece : AI;\n    const usePruning = options.usePruning !== undefined ? options.usePruning : true;\n    const starCells = options.starCells || [];\n\n    const workingBoard = cloneBoard(board);\n    const cols = workingBoard[0].length;\n    const validCols = getValidLocations(workingBoard);\n    const columnScores = {};\n    for (let c = 0; c < cols; c++) {\n      columnScores[c] = null;\n    }\n\n    if (validCols.length === 0) {\n      return {\n        bestCol: -1,\n        bestScore: 0,\n        columnScores,\n        nodes: 0,\n        elapsedMs: 0,\n        maxDepth,\n        usePruning,\n      };\n    }\n\n    const stats = { nodes: 0 };\n    const t0 = typeof performance !== 'undefined' ? performance.now() : Date.now();\n\n    let bestScore = -Infinity;\n    let bestCol = validCols[0];\n    let alpha = -Infinity;\n    const beta = Infinity;\n\n    for (let i = 0; i < validCols.length; i++) {\n      const col = validCols[i];\n      const row = getNextOpenRow(workingBoard, col);\n      workingBoard[row][col] = aiPiece;\n\n      const score = usePruning\n        ? minimaxAlphaBeta(workingBoard, 1, maxDepth, alpha, beta, false, aiPiece, stats, starCells)\n        : minimax(workingBoard, 1, maxDepth, false, aiPiece, stats, starCells);\n\n      workingBoard[row][col] = EMPTY;\n      columnScores[col] = score;\n\n      if (score > bestScore) {\n        bestScore = score;\n        bestCol = col;\n      }\n      if (usePruning && bestScore > alpha) {\n        alpha = bestScore;\n      }\n    }\n\n    const t1 = typeof performance !== 'undefined' ? performance.now() : Date.now();\n\n    return {\n      bestCol,\n      bestScore,\n      columnScores,\n      nodes: stats.nodes,\n      elapsedMs: Number((t1 - t0).toFixed(2)),\n      maxDepth,\n      usePruning,\n    };\n  }\n\n  /**\n   * Pilih langkah Bot AI berdasarkan tingkat kesulitan ('Easy', 'Medium', 'Hard')\n   * atau konfigurasi depth/pruning kustom.\n   */\n  function chooseMove(board, options = {}) {\n    const difficulty = options.difficulty || 'Hard';\n    const aiPiece = options.aiPiece !== undefined ? options.aiPiece : AI;\n    const starCells = options.starCells || [];\n    const preset = DIFFICULTY_PRESETS[difficulty] || DIFFICULTY_PRESETS.Hard;\n    const cols = board[0].length;\n\n    if (difficulty === 'Easy' && options.customDepth === undefined) {\n      const validCols = getValidLocations(board);\n      const columnScores = {};\n      for (let c = 0; c < cols; c++) {\n        columnScores[c] = null;\n      }\n      if (validCols.length === 0) {\n        return { bestCol: -1, bestScore: 0, columnScores, nodes: 0, elapsedMs: 0, maxDepth: 0, usePruning: false, isRandom: true };\n      }\n      const workingBoard = cloneBoard(board);\n      const t0 = typeof performance !== 'undefined' ? performance.now() : Date.now();\n      for (let i = 0; i < validCols.length; i++) {\n        const c = validCols[i];\n        const r = getNextOpenRow(workingBoard, c);\n        workingBoard[r][c] = aiPiece;\n        columnScores[c] = evaluateBoard(workingBoard, aiPiece, starCells);\n        workingBoard[r][c] = EMPTY;\n      }\n      const randomCol = validCols[Math.floor(Math.random() * validCols.length)];\n      const t1 = typeof performance !== 'undefined' ? performance.now() : Date.now();\n      return {\n        bestCol: randomCol,\n        bestScore: columnScores[randomCol],\n        columnScores,\n        nodes: validCols.length,\n        elapsedMs: Number((t1 - t0).toFixed(2)),\n        maxDepth: 0,\n        usePruning: false,\n        isRandom: true,\n      };\n    }\n\n    const maxDepth = options.customDepth !== undefined ? options.customDepth : preset.maxDepth;\n    const usePruning = options.usePruning !== undefined ? options.usePruning : preset.usePruning;\n\n    const result = findBestMove(board, { maxDepth, aiPiece, usePruning, starCells });\n    result.isRandom = false;\n    return result;\n  }\n\n  /**\n   * Jalankan perbandingan langsung Minimax Murni vs Alpha-Beta Pruning pada papan saat ini.\n   */\n  function compareAlgorithms(board, maxDepth = 4, aiPiece = AI, starCells = []) {\n    const pureMinimax = findBestMove(board, { maxDepth, aiPiece, usePruning: false, starCells });\n    const alphaBeta = findBestMove(board, { maxDepth, aiPiece, usePruning: true, starCells });\n\n    const reductionPct =\n      pureMinimax.nodes > 0\n        ? Number((((pureMinimax.nodes - alphaBeta.nodes) / pureMinimax.nodes) * 100).toFixed(1))\n        : 0;\n\n    return {\n      maxDepth,\n      pureMinimax,\n      alphaBeta,\n      reductionPct,\n      sameDecision: pureMinimax.bestCol === alphaBeta.bestCol,\n    };\n  }\n\n  return {\n    ROWS,\n    COLS,\n    WINDOW_LENGTH,\n    BLOCKED,\n    EMPTY,\n    HUMAN,\n    AI,\n    WIN_SCORE,\n    COLUMN_ORDER,\n    BOARD_SIZES,\n    EVAL_WEIGHTS,\n    DIFFICULTY_PRESETS,\n    getColumnOrder,\n    createBoard,\n    createSpecialSetup,\n    cloneBoard,\n    isValidLocation,\n    getValidLocations,\n    getNextOpenRow,\n    dropPiece,\n    getWinnerInfo,\n    checkWinner,\n    evaluateWindow,\n    analyzeBoard,\n    evaluateBoard,\n    minimax,\n    minimaxAlphaBeta,\n    findBestMove,\n    chooseMove,\n    compareAlgorithms,\n  };\n});\n\n\n(function () {\n  if (typeof google !== 'undefined' && google.colab && google.colab.kernel) {\n    window.ConnectFourBridge = {\n      label: 'Engine: Python Kernel Aktif (Google Colab • google.colab.output)',\n      async chooseMove(board, options) {\n        try {\n          const res = await google.colab.kernel.invokeFunction(\n            'connect4.bot_move',\n            [\n              JSON.stringify(board),\n              options.difficulty || 'Hard',\n              options.aiPiece !== undefined ? options.aiPiece : 2,\n              options.usePruning !== undefined ? options.usePruning : null,\n              options.customDepth !== undefined ? options.customDepth : null,\n              JSON.stringify(options.starCells || []),\n            ],\n            {}\n          );\n          const data = res && res.data && res.data['application/json'];\n          if (data) return data;\n        } catch (err) {\n          console.warn('Colab Python bridge fallback ke JS:', err);\n        }\n        return window.ConnectFourAI.chooseMove(board, options);\n      },\n      async compareAlgorithms(board, depth, aiPiece, starCells) {\n        try {\n          const res = await google.colab.kernel.invokeFunction(\n            'connect4.compare',\n            [JSON.stringify(board), depth, aiPiece, JSON.stringify(starCells || [])],\n            {}\n          );\n          const data = res && res.data && res.data['application/json'];\n          if (data) return data;\n        } catch (err) {\n          console.warn('Colab Python compare fallback ke JS:', err);\n        }\n        return window.ConnectFourAI.compareAlgorithms(board, depth, aiPiece, starCells || []);\n      },\n    };\n  }\n})();\n\n(function () {\n  'use strict';\n\n  const AI_LIB = window.ConnectFourAI;\n  const { ROWS, COLS, BLOCKED, EMPTY, HUMAN, AI, WIN_SCORE } = AI_LIB;\n\n  const state = {\n    view: 'home', // 'home' | 'setup' | 'game'\n    gameMode: 'hvb', // 'hvb' | 'bvb'\n    boardMode: 'classic', // 'classic' | 'obstacle' | 'star'\n    boardSize: '6x7', // '6x7' | '7x8' | '7x9'\n    tileShape: 'circle', // 'circle' | 'square'\n    difficulty: 'Hard', // 'Easy' | 'Medium' | 'Hard' | 'Custom'\n    firstChoice: '1', // '1' | '2' | 'random'\n    resolvedFirstPlayer: HUMAN,\n    randomStartNote: '',\n    board: AI_LIB.createBoard(),\n    starCells: [],\n    currentPlayer: HUMAN,\n    gameOver: false,\n    winner: null,\n    winCells: [],\n    lastMove: null,\n    lastAiResult: null,\n    moveCount: 0,\n    moveLog: [],\n    history: [],\n    compareTarget: null,\n    isThinking: false,\n    autoPlayPaused: false,\n    matchGeneration: 0,\n    autoPlayTimer: null,\n    pendingBotTimer: null,\n    postGameTimer: null,\n    hoverCol: -1,\n  };\n\n  const modalCloseTimers = new WeakMap();\n\n  async function requestBotMove(board, options) {\n    const bridge = window.ConnectFourBridge;\n    if (bridge && typeof bridge.chooseMove === 'function') {\n      return await bridge.chooseMove(board, options);\n    }\n    return AI_LIB.chooseMove(board, options);\n  }\n\n  async function requestCompare(board, depth, aiPiece, starCells) {\n    const bridge = window.ConnectFourBridge;\n    if (bridge && typeof bridge.compareAlgorithms === 'function') {\n      return await bridge.compareAlgorithms(board, depth, aiPiece, starCells);\n    }\n    return AI_LIB.compareAlgorithms(board, depth, aiPiece, starCells);\n  }\n\n  const els = {\n    // Views\n    homeView: document.getElementById('homeView'),\n    setupView: document.getElementById('setupView'),\n    gameView: document.getElementById('gameView'),\n    arenaShell: document.getElementById('arenaShell'),\n\n    // Home Menu\n    goSetupBtn: document.getElementById('goSetupBtn'),\n    openRulesBtn: document.getElementById('openRulesBtn'),\n    openInfoBtn: document.getElementById('openInfoBtn'),\n\n    // Setup Menu\n    backHomeBtn: document.getElementById('backHomeBtn'),\n    setupPreviewBadge: document.getElementById('setupPreviewBadge'),\n    modeChoiceGrid: document.getElementById('modeChoiceGrid'),\n    boardModeGrid: document.getElementById('boardModeGrid'),\n    boardSizeGrid: document.getElementById('boardSizeGrid'),\n    tileShapeGrid: document.getElementById('tileShapeGrid'),\n    diffChoiceGrid: document.getElementById('diffChoiceGrid'),\n    firstChoiceGrid: document.getElementById('firstChoiceGrid'),\n    firstP1Title: document.getElementById('firstP1Title'),\n    customConfigBox: document.getElementById('customConfigBox'),\n    algorithmSelect: document.getElementById('algorithmSelect'),\n    depthSelect: document.getElementById('depthSelect'),\n    startGameBtn: document.getElementById('startGameBtn'),\n\n    // Modals\n    rulesModal: document.getElementById('rulesModal'),\n    closeRulesBtn: document.getElementById('closeRulesBtn'),\n    infoModal: document.getElementById('infoModal'),\n    closeInfoBtn: document.getElementById('closeInfoBtn'),\n    modalLegendList: document.getElementById('modalLegendList'),\n\n    // Arena\n    arenaPill: document.getElementById('arenaPill'),\n    backMenuBtn: document.getElementById('backMenuBtn'),\n    backSetupBtn: document.getElementById('backSetupBtn'),\n    activeConfigBadge: document.getElementById('activeConfigBadge'),\n    turnCounterBadge: document.getElementById('turnCounterBadge'),\n    undoBtn: document.getElementById('undoBtn'),\n    resetBtn: document.getElementById('resetBtn'),\n    bvbBar: document.getElementById('bvbBar'),\n    botStepBtn: document.getElementById('botStepBtn'),\n    autoPlayBtn: document.getElementById('autoPlayBtn'),\n    statusBanner: document.getElementById('statusBanner'),\n    statusDot: document.getElementById('statusDot'),\n    statusText: document.getElementById('statusText'),\n    thinkingDots: document.getElementById('thinkingDots'),\n    lastMovePill: document.getElementById('lastMovePill'),\n    hoverPreviewRow: document.getElementById('hoverPreviewRow'),\n    colSelectors: document.getElementById('colSelectors'),\n    boardGrid: document.getElementById('boardGrid'),\n    p1Swatch: document.getElementById('p1Swatch'),\n    p2Swatch: document.getElementById('p2Swatch'),\n    p1LegendLabel: document.getElementById('p1LegendLabel'),\n    p2LegendLabel: document.getElementById('p2LegendLabel'),\n    specialTileLegend: document.getElementById('specialTileLegend'),\n\n    // Post-Game Analysis\n    postGamePanel: document.getElementById('postGamePanel'),\n    postGameBanner: document.getElementById('postGameBanner'),\n    postGameTitle: document.getElementById('postGameTitle'),\n    postGameSub: document.getElementById('postGameSub'),\n    playAgainBtn: document.getElementById('playAgainBtn'),\n    postMenuBtn: document.getElementById('postMenuBtn'),\n    statTotalMoves: document.getElementById('statTotalMoves'),\n    statBotMoves: document.getElementById('statBotMoves'),\n    statAvgNodes: document.getElementById('statAvgNodes'),\n    statTotalTime: document.getElementById('statTotalTime'),\n    postGameTableBody: document.getElementById('postGameTableBody'),\n    legendTotalBadge: document.getElementById('legendTotalBadge'),\n    heuristicGrid: document.getElementById('heuristicGrid'),\n    compareDepthSelect: document.getElementById('compareDepthSelect'),\n    compareBtn: document.getElementById('compareBtn'),\n    compareContextText: document.getElementById('compareContextText'),\n    compareResult: document.getElementById('compareResult'),\n  };\n\n  function getAiConfig() {\n    if (state.difficulty === 'Easy') {\n      return { difficulty: 'Easy', usePruning: false, maxDepth: 0, label: 'Easy • Random' };\n    }\n    if (state.difficulty === 'Medium') {\n      return { difficulty: 'Medium', usePruning: false, maxDepth: 3, label: 'Medium • Minimax d=3' };\n    }\n    if (state.difficulty === 'Hard') {\n      return { difficulty: 'Hard', usePruning: true, maxDepth: 5, label: 'Hard • α-β d=5' };\n    }\n    const usePruning = els.algorithmSelect.value === 'alphabeta';\n    const maxDepth = Number(els.depthSelect.value);\n    const algoTxt = usePruning ? 'α-β' : 'Minimax';\n    return {\n      difficulty: 'Custom',\n      usePruning,\n      maxDepth,\n      label: `Custom • ${algoTxt} d=${maxDepth}`,\n    };\n  }\n\n  function formatScore(score) {\n    if (score === null || score === undefined) return '-';\n    if (score >= WIN_SCORE - 100) return `+${score} (Win)`;\n    if (score <= -(WIN_SCORE - 100)) return `${score} (Lose)`;\n    return score > 0 ? `+${score}` : `${score}`;\n  }\n\n  function pulsePickedCard(cardEl) {\n    if (!cardEl) return;\n    cardEl.classList.remove('just-picked');\n    if (typeof cardEl.offsetWidth === 'number') {\n      void cardEl.offsetWidth;\n    }\n    cardEl.classList.add('just-picked');\n  }\n\n  function openModal(modalEl) {\n    if (!modalEl) return;\n    const pending = modalCloseTimers.get(modalEl);\n    if (pending) {\n      clearTimeout(pending);\n      modalCloseTimers.delete(modalEl);\n    }\n    modalEl.classList.remove('is-closing');\n    modalEl.style.display = 'flex';\n  }\n\n  function closeModal(modalEl) {\n    if (!modalEl || modalEl.style.display === 'none' || modalEl.classList.contains('is-closing')) {\n      return;\n    }\n    modalEl.classList.add('is-closing');\n    const timer = setTimeout(() => {\n      modalCloseTimers.delete(modalEl);\n      modalEl.style.display = 'none';\n      modalEl.classList.remove('is-closing');\n    }, 180);\n    modalCloseTimers.set(modalEl, timer);\n  }\n\n  function syncMenuUI() {\n    els.modeChoiceGrid.querySelectorAll('.choice-card').forEach((btn) => {\n      btn.classList.toggle('active', btn.dataset.mode === state.gameMode);\n    });\n    els.boardModeGrid.querySelectorAll('.choice-card').forEach((btn) => {\n      btn.classList.toggle('active', btn.dataset.bmode === state.boardMode);\n    });\n    if (els.boardSizeGrid) {\n      els.boardSizeGrid.querySelectorAll('.size-btn').forEach((btn) => {\n        btn.classList.toggle('active', btn.dataset.size === state.boardSize);\n      });\n    }\n    els.tileShapeGrid.querySelectorAll('.shape-btn').forEach((btn) => {\n      btn.classList.toggle('active', btn.dataset.shape === state.tileShape);\n    });\n    els.diffChoiceGrid.querySelectorAll('.choice-card').forEach((btn) => {\n      btn.classList.toggle('active', btn.dataset.diff === state.difficulty);\n    });\n    els.firstChoiceGrid.querySelectorAll('.choice-card').forEach((btn) => {\n      btn.classList.toggle('active', btn.dataset.first === state.firstChoice);\n    });\n\n    els.firstP1Title.textContent = state.gameMode === 'bvb' ? 'Bot Coral' : 'Manusia';\n    els.customConfigBox.style.display = state.difficulty === 'Custom' ? 'block' : 'none';\n\n    const cfg = getAiConfig();\n    const modeTag = state.boardMode === 'classic' ? '' : state.boardMode === 'obstacle' ? ' • Batu' : ' • Star';\n    const sizeTag = state.boardSize === '6x7' ? '' : ` • ${state.boardSize}`;\n    els.setupPreviewBadge.textContent = `${cfg.label}${modeTag}${sizeTag}`;\n    els.activeConfigBadge.textContent = `${cfg.label}${modeTag}${sizeTag}`;\n  }\n\n  function clearTimers() {\n    if (state.pendingBotTimer) {\n      clearTimeout(state.pendingBotTimer);\n      state.pendingBotTimer = null;\n    }\n    if (state.autoPlayTimer) {\n      clearTimeout(state.autoPlayTimer);\n      state.autoPlayTimer = null;\n    }\n    if (state.postGameTimer) {\n      clearTimeout(state.postGameTimer);\n      state.postGameTimer = null;\n    }\n    state.isThinking = false;\n  }\n\n  function retriggerStagePop(viewEl) {\n    const stage = viewEl && viewEl.querySelector ? viewEl.querySelector('.animate-pop-in') : null;\n    if (!stage) return;\n    stage.classList.remove('animate-pop-in');\n    if (typeof stage.offsetWidth === 'number') {\n      void stage.offsetWidth;\n    }\n    stage.classList.add('animate-pop-in');\n  }\n\n  function switchView(viewName) {\n    state.view = viewName;\n    els.homeView.style.display = viewName === 'home' ? 'flex' : 'none';\n    els.setupView.style.display = viewName === 'setup' ? 'flex' : 'none';\n    els.gameView.style.display = viewName === 'game' ? 'block' : 'none';\n\n    if (viewName === 'home') retriggerStagePop(els.homeView);\n    else if (viewName === 'setup') retriggerStagePop(els.setupView);\n  }\n\n  function renderScoringLegend(targetEl, info, winner, includeStarRow = false) {\n    function renderMiniTiles(dots) {\n      return (\n        '<span class=\"mini-tiles\">' +\n        dots.map((d) => `<span class=\"mini-tile\"><span class=\"mini-dot ${d}\"></span></span>`).join('') +\n        '</span>'\n      );\n    }\n\n    const rows = [\n      {\n        dots: ['ai', 'ai', 'ai', 'ai'],\n        label: '4-in-a-Row AI',\n        count: winner === AI ? 'Menang' : info ? '0x' : 'Terminal',\n        pts: '+100000',\n      },\n      {\n        dots: ['ai', 'ai', 'ai', 'empty'],\n        label: '3 AI + 1 Kosong',\n        count: info ? `${info.aiThree}x` : 'per pola',\n        pts: '+12',\n      },\n      {\n        dots: ['ai', 'ai', 'empty', 'empty'],\n        label: '2 AI + 2 Kosong',\n        count: info ? `${info.aiTwo}x` : 'per pola',\n        pts: '+3',\n      },\n      {\n        dots: ['opp', 'opp', 'opp', 'empty'],\n        label: 'Ancaman 3 Lawan',\n        count: info ? `${info.oppThree}x` : 'per pola',\n        pts: '-80',\n      },\n      {\n        dots: ['opp', 'opp', 'empty', 'empty'],\n        label: '2 Lawan + 2 Kosong',\n        count: info ? `${info.oppTwo}x` : 'per pola',\n        pts: '-2',\n      },\n      {\n        dots: ['center'],\n        label: 'Kontrol Kolom Tengah',\n        count: info ? `AI ${info.centerAi}:${info.centerOpp}` : 'per selisih',\n        pts: info ? `${info.centerScore >= 0 ? '+' : ''}${info.centerScore}` : '+4',\n      },\n    ];\n\n    if (includeStarRow) {\n      rows.push({\n        dots: ['star'],\n        label: 'Bonus Petak Star (★)',\n        count: info ? `AI ${info.starAi}:${info.starOpp}` : 'mode Star',\n        pts: info ? `${info.starScore >= 0 ? '+' : ''}${info.starScore}` : '+6',\n      });\n    }\n\n    targetEl.innerHTML = '';\n    rows.forEach((row) => {\n      const el = document.createElement('div');\n      el.className = 'legend-score-row';\n      el.innerHTML = `\n        <div class=\"legend-pattern\">\n          ${renderMiniTiles(row.dots)}\n          <span>${row.label}</span>\n        </div>\n        <div class=\"legend-right\">\n          <span class=\"legend-live-count\">${row.count}</span>\n          <span class=\"legend-points\">${row.pts}</span>\n        </div>\n      `;\n      targetEl.appendChild(el);\n    });\n  }\n\n  function getWinningIndex(r, c) {\n    for (let i = 0; i < state.winCells.length; i++) {\n      if (state.winCells[i][0] === r && state.winCells[i][1] === c) return i;\n    }\n    return -1;\n  }\n\n  function isStarCell(r, c) {\n    for (let i = 0; i < state.starCells.length; i++) {\n      if (state.starCells[i][0] === r && state.starCells[i][1] === c) return true;\n    }\n    return false;\n  }\n\n  function saveSnapshot() {\n    state.history.push({\n      board: AI_LIB.cloneBoard(state.board),\n      starCells: state.starCells.map((p) => p.slice()),\n      currentPlayer: state.currentPlayer,\n      gameOver: state.gameOver,\n      winner: state.winner,\n      winCells: state.winCells.slice(),\n      lastMove: state.lastMove ? { ...state.lastMove } : null,\n      lastAiResult: state.lastAiResult ? { ...state.lastAiResult, columnScores: { ...state.lastAiResult.columnScores } } : null,\n      moveCount: state.moveCount,\n      moveLog: state.moveLog.slice(),\n      compareTarget: state.compareTarget\n        ? {\n            board: AI_LIB.cloneBoard(state.compareTarget.board),\n            aiPiece: state.compareTarget.aiPiece,\n            step: state.compareTarget.step,\n            starCells: state.compareTarget.starCells.map((p) => p.slice()),\n          }\n        : null,\n    });\n  }\n\n  function restoreSnapshot(snap) {\n    state.board = AI_LIB.cloneBoard(snap.board);\n    state.starCells = snap.starCells.map((p) => p.slice());\n    state.currentPlayer = snap.currentPlayer;\n    state.gameOver = snap.gameOver;\n    state.winner = snap.winner;\n    state.winCells = snap.winCells.slice();\n    state.lastMove = snap.lastMove ? { ...snap.lastMove } : null;\n    state.lastAiResult = snap.lastAiResult;\n    state.moveCount = snap.moveCount;\n    state.moveLog = snap.moveLog.slice();\n    state.compareTarget = snap.compareTarget\n      ? {\n          board: AI_LIB.cloneBoard(snap.compareTarget.board),\n          aiPiece: snap.compareTarget.aiPiece,\n          step: snap.compareTarget.step,\n          starCells: snap.compareTarget.starCells.map((p) => p.slice()),\n        }\n      : null;\n  }\n\n  function renderBoard() {\n    const isBvb = state.gameMode === 'bvb';\n    const humanCanClick = !state.gameOver && !state.isThinking && !isBvb && state.currentPlayer === HUMAN;\n    const isSquare = state.tileShape === 'square';\n    const rows = state.board.length;\n    const cols = state.board[0].length;\n\n    els.boardGrid.classList.toggle('shape-square', isSquare);\n    els.p1Swatch.classList.toggle('shape-square', isSquare);\n    els.p2Swatch.classList.toggle('shape-square', isSquare);\n    els.statusDot.classList.toggle('shape-square', isSquare);\n\n    [els.hoverPreviewRow, els.colSelectors, els.boardGrid].forEach((el) => {\n      if (!el || !el.style) return;\n      el.style.setProperty('--board-cols', String(cols));\n      el.style.setProperty('--board-rows', String(rows));\n    });\n\n    // Ghost Hover Preview Row\n    els.hoverPreviewRow.innerHTML = '';\n    for (let c = 0; c < cols; c++) {\n      const slot = document.createElement('div');\n      slot.className = 'ghost-slot';\n      const ghost = document.createElement('div');\n      ghost.className = `ghost-disc${isSquare ? ' shape-square' : ''}`;\n      ghost.dataset.col = String(c);\n      slot.appendChild(ghost);\n      els.hoverPreviewRow.appendChild(slot);\n    }\n\n    // Column selector buttons\n    els.colSelectors.innerHTML = '';\n    for (let c = 0; c < cols; c++) {\n      const btn = document.createElement('button');\n      btn.type = 'button';\n      btn.className = 'col-btn';\n      const valid = AI_LIB.isValidLocation(state.board, c);\n      if (state.lastAiResult && state.lastAiResult.bestCol === c) {\n        btn.classList.add('col-best');\n      }\n      btn.disabled = !humanCanClick || !valid;\n      btn.textContent = `K${c}`;\n      btn.title = `Jatuhkan ke Kolom ${c}`;\n      btn.addEventListener('mouseenter', () => setHoverCol(c));\n      btn.addEventListener('mouseleave', () => setHoverCol(-1));\n      btn.addEventListener('click', () => handleHumanMove(c));\n      els.colSelectors.appendChild(btn);\n    }\n\n    // Dynamic Grid (6x7 / 7x8 / 7x9)\n    els.boardGrid.innerHTML = '';\n    for (let r = 0; r < rows; r++) {\n      for (let c = 0; c < cols; c++) {\n        const cell = document.createElement('div');\n        cell.className = 'cell';\n        cell.dataset.row = String(r);\n        cell.dataset.col = String(c);\n\n        const val = state.board[r][c];\n        const blocked = val === BLOCKED;\n        const star = isStarCell(r, c);\n\n        if (blocked) {\n          cell.classList.add('is-blocked', 'disabled');\n        } else if (!humanCanClick || !AI_LIB.isValidLocation(state.board, c)) {\n          cell.classList.add('disabled');\n        }\n\n        if (star) {\n          cell.classList.add('is-star');\n          if (val === HUMAN || val === AI) cell.classList.add('has-disc');\n        }\n\n        if (state.lastMove && state.lastMove.row === r && state.lastMove.col === c) {\n          cell.classList.add('last-move');\n        }\n        const winIdx = getWinningIndex(r, c);\n        if (winIdx !== -1) {\n          cell.classList.add('win-cell');\n          cell.style.setProperty('--win-idx', String(winIdx));\n        }\n\n        if (val === HUMAN || val === AI) {\n          const disc = document.createElement('div');\n          disc.className = `disc ${val === HUMAN ? 'p1' : 'p2'}`;\n          if (state.lastMove && state.lastMove.row === r && state.lastMove.col === c && state.lastMove.animate) {\n            disc.classList.add('drop-anim');\n            disc.style.setProperty('--drop-rows', String(r + 1));\n          }\n          cell.appendChild(disc);\n        }\n\n        if (!blocked) {\n          cell.addEventListener('mouseenter', () => setHoverCol(c));\n          cell.addEventListener('mouseleave', () => setHoverCol(-1));\n          cell.addEventListener('click', () => {\n            if (humanCanClick) handleHumanMove(c);\n          });\n        }\n        els.boardGrid.appendChild(cell);\n      }\n    }\n\n    if (state.lastMove) {\n      state.lastMove.animate = false;\n    }\n    applyHoverVisuals();\n  }\n\n  function applyHoverVisuals() {\n    const isBvb = state.gameMode === 'bvb';\n    const humanCanClick = !state.gameOver && !state.isThinking && !isBvb && state.currentPlayer === HUMAN;\n    const col = state.hoverCol;\n    const landingRow =\n      humanCanClick && col >= 0 && AI_LIB.isValidLocation(state.board, col)\n        ? AI_LIB.getNextOpenRow(state.board, col)\n        : -1;\n\n    els.hoverPreviewRow.querySelectorAll('.ghost-disc').forEach((g, idx) => {\n      g.classList.toggle('visible', idx === col && landingRow !== -1);\n    });\n\n    els.colSelectors.querySelectorAll('.col-btn').forEach((btn, idx) => {\n      btn.classList.toggle('col-hover', idx === col && landingRow !== -1);\n    });\n\n    els.boardGrid.querySelectorAll('.cell').forEach((cell) => {\n      const r = Number(cell.dataset.row);\n      const c = Number(cell.dataset.col);\n      const isColHover = c === col && landingRow !== -1 && state.board[r][c] === EMPTY;\n      cell.classList.toggle('col-active-hover', isColHover && r !== landingRow);\n      cell.classList.toggle('target-landing', isColHover && r === landingRow);\n    });\n  }\n\n  function setHoverCol(col) {\n    if (state.hoverCol === col) return;\n    state.hoverCol = col;\n    applyHoverVisuals();\n  }\n\n  function renderArenaHeaderAndStatus() {\n    const isBvb = state.gameMode === 'bvb';\n    const modeLabel = state.boardMode === 'obstacle' ? 'OBSTACLE' : state.boardMode === 'star' ? 'STAR +6' : 'ARENA';\n    els.arenaPill.textContent = isBvb ? `BOT VS BOT • ${modeLabel}` : `${modeLabel} MAIN`;\n\n    els.p1LegendLabel.textContent = isBvb ? 'Bot Coral (d=3)' : 'Manusia';\n    els.p2LegendLabel.textContent = isBvb ? 'Bot Kuning' : 'Bot AI';\n\n    if (state.boardMode === 'obstacle') {\n      els.specialTileLegend.style.display = 'inline-flex';\n      els.specialTileLegend.textContent = '✕ Batu Rintangan';\n    } else if (state.boardMode === 'star') {\n      els.specialTileLegend.style.display = 'inline-flex';\n      els.specialTileLegend.textContent = '★ Bonus +6';\n    } else {\n      els.specialTileLegend.style.display = 'none';\n    }\n\n    els.bvbBar.style.display = isBvb && !state.gameOver ? 'flex' : 'none';\n    els.autoPlayBtn.textContent = state.autoPlayPaused ? 'Lanjut Auto-Play' : 'Jeda Auto-Play';\n\n    els.turnCounterBadge.textContent = `#${state.moveCount}`;\n    els.undoBtn.disabled = state.history.length === 0 || state.isThinking;\n\n    if (state.lastAiResult) {\n      els.lastMovePill.style.display = 'inline-block';\n      els.lastMovePill.textContent = `Kol ${state.lastAiResult.bestCol} (${formatScore(state.lastAiResult.bestScore)})`;\n    } else {\n      els.lastMovePill.style.display = 'none';\n    }\n\n    els.statusBanner.className = 'status-banner';\n    els.statusDot.className = `status-dot${state.tileShape === 'square' ? ' shape-square' : ''}`;\n    els.thinkingDots.classList.remove('active');\n\n    if (state.gameOver) {\n      if (state.winner === 'DRAW') {\n        els.statusBanner.classList.add('game-draw');\n        els.statusDot.classList.add('neutral');\n        els.statusText.textContent = 'Permainan Berakhir SERI (Papan Penuh)';\n      } else {\n        els.statusBanner.classList.add('game-win');\n        els.statusDot.classList.add(state.winner === HUMAN ? 'p1' : 'p2');\n        if (isBvb) {\n          els.statusText.textContent =\n            state.winner === HUMAN ? 'Bot Coral (Pemain 1) MENANG!' : 'Bot Kuning (Pemain 2) MENANG!';\n        } else {\n          els.statusText.textContent =\n            state.winner === HUMAN ? 'Selamat! Anda MENANG!' : 'Bot AI MENANG!';\n        }\n      }\n      return;\n    }\n\n    if (state.isThinking) {\n      els.statusBanner.classList.add(state.currentPlayer === HUMAN ? 'turn-p1' : 'turn-p2');\n      els.statusDot.classList.add(state.currentPlayer === HUMAN ? 'p1' : 'p2');\n      els.statusText.textContent = 'Bot AI sedang berpikir';\n      els.thinkingDots.classList.add('active');\n      return;\n    }\n\n    const prefix = state.moveCount === 0 && state.randomStartNote ? `${state.randomStartNote} ` : '';\n    const cols = state.board[0].length;\n    if (state.currentPlayer === HUMAN) {\n      els.statusBanner.classList.add('turn-p1');\n      els.statusDot.classList.add('p1');\n      els.statusText.textContent = isBvb\n        ? `${prefix}Giliran Bot Coral...`\n        : `${prefix}Giliran Anda — Pilih Kolom 0–${cols - 1}`;\n    } else {\n      els.statusBanner.classList.add('turn-p2');\n      els.statusDot.classList.add('p2');\n      els.statusText.textContent = isBvb\n        ? `${prefix}Giliran Bot Kuning...`\n        : `${prefix}Giliran Bot AI...`;\n    }\n  }\n\n  function renderPostGameAnalysis(showPanel) {\n    if (!showPanel || !state.gameOver) {\n      els.arenaShell.classList.remove('has-analysis');\n      els.postGamePanel.style.display = 'none';\n      return;\n    }\n\n    els.arenaShell.classList.add('has-analysis');\n    els.postGamePanel.style.display = 'block';\n\n    const isBvb = state.gameMode === 'bvb';\n    els.postGameBanner.className = 'postgame-banner';\n    if (state.winner === 'DRAW') {\n      els.postGameBanner.classList.add('is-draw');\n      els.postGameTitle.textContent = 'Hasil Akhir: SERI!';\n      els.postGameSub.textContent = `Papan penuh setelah ${state.moveCount} langkah tanpa 4-in-a-row.`;\n    } else if (state.winner === HUMAN) {\n      els.postGameTitle.textContent = isBvb ? 'Bot Coral (Pemain 1) Menang!' : 'Selamat, Anda Menang!';\n      els.postGameSub.textContent = `4 bidak terhubung dalam total ${state.moveCount} langkah permainan.`;\n    } else {\n      els.postGameBanner.classList.add('is-bot-win');\n      els.postGameTitle.textContent = isBvb ? 'Bot Kuning (Pemain 2) Menang!' : 'Bot AI (Kuning) Menang!';\n      els.postGameSub.textContent = `Bot AI mengunci kemenangan pada langkah ke-${state.moveCount}.`;\n    }\n\n    const botEntries = state.moveLog.filter((m) => m.isBot);\n    const totalNodes = botEntries.reduce((acc, m) => acc + m.nodes, 0);\n    const totalTime = botEntries.reduce((acc, m) => acc + m.elapsedMs, 0);\n    const avgNodes = botEntries.length > 0 ? Math.round(totalNodes / botEntries.length) : 0;\n\n    els.statTotalMoves.textContent = String(state.moveCount);\n    els.statBotMoves.textContent = String(botEntries.length);\n    els.statAvgNodes.textContent = avgNodes.toLocaleString('id-ID');\n    els.statTotalTime.textContent = `${totalTime.toFixed(1)} ms`;\n\n    els.postGameTableBody.innerHTML = '';\n    state.moveLog.forEach((entry, idx) => {\n      const tr = document.createElement('tr');\n      tr.style.setProperty('--row-idx', String(idx));\n      if (entry.isBot) tr.classList.add('row-bot');\n      if (idx === state.moveLog.length - 1 && state.winner !== 'DRAW') tr.classList.add('row-win');\n      tr.innerHTML = `\n        <td>#${entry.step}</td>\n        <td>${entry.actor}</td>\n        <td>Kol ${entry.col} (R${entry.row})</td>\n        <td>${entry.scoreText}</td>\n        <td>${entry.isBot ? entry.nodes.toLocaleString('id-ID') : '-'}</td>\n        <td>${entry.isBot ? `${entry.elapsedMs} ms` : '-'}</td>\n      `;\n      els.postGameTableBody.appendChild(tr);\n    });\n\n    const finalInfo = AI_LIB.analyzeBoard(state.board, AI, state.starCells);\n    els.legendTotalBadge.textContent = `Total: ${formatScore(finalInfo.totalScore)}`;\n    renderScoringLegend(els.heuristicGrid, finalInfo, state.winner, state.boardMode === 'star');\n\n    if (state.compareTarget) {\n      els.compareContextText.textContent =\n        `Bandingkan jumlah node pada langkah #${state.compareTarget.step} (sebelum langkah penutup).`;\n    }\n  }\n\n  function renderAll(showPostGameNow = false) {\n    renderArenaHeaderAndStatus();\n    renderBoard();\n    renderPostGameAnalysis(state.gameOver && showPostGameNow);\n  }\n\n  function executeDrop(col, player, aiResult) {\n    const row = AI_LIB.getNextOpenRow(state.board, col);\n    if (row === -1) return false;\n\n    const preMoveBoard = AI_LIB.cloneBoard(state.board);\n    saveSnapshot();\n    AI_LIB.dropPiece(state.board, row, col, player);\n    state.moveCount += 1;\n    state.lastMove = { row, col, animate: true };\n\n    if (aiResult) {\n      state.lastAiResult = aiResult;\n    }\n\n    state.compareTarget = {\n      board: preMoveBoard,\n      aiPiece: player === HUMAN && state.gameMode === 'hvb' ? AI : player,\n      step: state.moveCount,\n      starCells: state.starCells.map((p) => p.slice()),\n    };\n\n    const isBvb = state.gameMode === 'bvb';\n    const isBot = isBvb || player === AI;\n    const actor =\n      player === HUMAN ? (isBvb ? 'Bot Coral' : 'Manusia') : isBvb ? 'Bot Kuning' : 'Bot AI';\n\n    const scoreText = aiResult\n      ? formatScore(aiResult.bestScore)\n      : formatScore(AI_LIB.evaluateBoard(state.board, AI, state.starCells));\n\n    state.moveLog.push({\n      step: state.moveCount,\n      actor,\n      col,\n      row,\n      isBot,\n      scoreText,\n      nodes: aiResult ? aiResult.nodes : 0,\n      elapsedMs: aiResult ? aiResult.elapsedMs : 0,\n    });\n\n    const winInfo = AI_LIB.getWinnerInfo(state.board);\n    if (winInfo.winner !== null) {\n      state.gameOver = true;\n      state.winner = winInfo.winner;\n      state.winCells = winInfo.cells;\n      clearTimers();\n      renderAll(false);\n      state.postGameTimer = setTimeout(() => {\n        state.postGameTimer = null;\n        renderPostGameAnalysis(true);\n      }, 360);\n      return true;\n    }\n\n    state.currentPlayer = player === HUMAN ? AI : HUMAN;\n    renderAll(false);\n    return true;\n  }\n\n  function handleHumanMove(col) {\n    if (state.view !== 'game' || state.gameOver || state.isThinking) return;\n    if (state.gameMode !== 'hvb' || state.currentPlayer !== HUMAN) return;\n    if (!AI_LIB.isValidLocation(state.board, col)) return;\n\n    const moved = executeDrop(col, HUMAN, null);\n    if (moved && !state.gameOver) {\n      scheduleBotTurn();\n    }\n  }\n\n  function scheduleBotTurn() {\n    const gen = state.matchGeneration;\n    state.isThinking = true;\n    renderArenaHeaderAndStatus();\n    applyHoverVisuals();\n\n    state.pendingBotTimer = setTimeout(async () => {\n      state.pendingBotTimer = null;\n      if (state.gameOver || gen !== state.matchGeneration) {\n        state.isThinking = false;\n        renderArenaHeaderAndStatus();\n        return;\n      }\n      await runSingleBotStep(gen);\n    }, 60);\n  }\n\n  async function runSingleBotStep(expectedGen = state.matchGeneration) {\n    if (state.gameOver || expectedGen !== state.matchGeneration) return;\n    const isBvb = state.gameMode === 'bvb';\n    const piece = state.currentPlayer;\n\n    state.isThinking = true;\n    renderArenaHeaderAndStatus();\n\n    const boardCopy = AI_LIB.cloneBoard(state.board);\n    const starCopy = state.starCells.map((p) => p.slice());\n    let res;\n    if (isBvb && piece === HUMAN) {\n      res = await requestBotMove(boardCopy, {\n        difficulty: 'Medium',\n        aiPiece: HUMAN,\n        usePruning: false,\n        customDepth: 3,\n        starCells: starCopy,\n      });\n    } else {\n      const cfg = getAiConfig();\n      res = await requestBotMove(boardCopy, {\n        difficulty: cfg.difficulty,\n        aiPiece: piece,\n        usePruning: cfg.usePruning,\n        customDepth: cfg.difficulty === 'Custom' ? cfg.maxDepth : undefined,\n        starCells: starCopy,\n      });\n    }\n\n    if (expectedGen !== state.matchGeneration || state.gameOver) {\n      state.isThinking = false;\n      return;\n    }\n\n    state.isThinking = false;\n    if (res && res.bestCol !== -1) {\n      executeDrop(res.bestCol, piece, res);\n    } else {\n      renderArenaHeaderAndStatus();\n    }\n  }\n\n  async function startAutoPlayLoop() {\n    if (state.gameMode !== 'bvb' || state.gameOver) return;\n    if (state.autoPlayTimer) {\n      clearTimeout(state.autoPlayTimer);\n      state.autoPlayTimer = null;\n    }\n    const gen = state.matchGeneration;\n    state.autoPlayPaused = false;\n    renderArenaHeaderAndStatus();\n\n    while (!state.gameOver && !state.autoPlayPaused && gen === state.matchGeneration && state.view === 'game') {\n      await runSingleBotStep(gen);\n      if (state.gameOver || state.autoPlayPaused || gen !== state.matchGeneration || state.view !== 'game') {\n        break;\n      }\n      await new Promise((resolve) => {\n        state.autoPlayTimer = setTimeout(() => {\n          state.autoPlayTimer = null;\n          resolve();\n        }, 420);\n      });\n    }\n  }\n\n  function resetGame() {\n    state.matchGeneration += 1;\n    clearTimers();\n\n    const dims = (AI_LIB.BOARD_SIZES && AI_LIB.BOARD_SIZES[state.boardSize]) || { rows: ROWS, cols: COLS };\n    const setup = AI_LIB.createSpecialSetup(state.boardMode, dims.rows, dims.cols);\n    state.board = setup.board;\n    state.starCells = setup.starCells;\n\n    if (state.firstChoice === 'random') {\n      state.resolvedFirstPlayer = Math.random() < 0.5 ? HUMAN : AI;\n      state.randomStartNote =\n        state.resolvedFirstPlayer === HUMAN\n          ? state.gameMode === 'bvb' ? '[Acak: Coral duluan]' : '[Acak: Anda duluan]'\n          : state.gameMode === 'bvb' ? '[Acak: Kuning duluan]' : '[Acak: Bot duluan]';\n    } else {\n      state.resolvedFirstPlayer = Number(state.firstChoice);\n      state.randomStartNote = '';\n    }\n\n    state.currentPlayer = state.resolvedFirstPlayer;\n    state.gameOver = false;\n    state.winner = null;\n    state.winCells = [];\n    state.lastMove = null;\n    state.lastAiResult = null;\n    state.moveCount = 0;\n    state.moveLog = [];\n    state.history = [];\n    state.compareTarget = null;\n    state.hoverCol = -1;\n    state.autoPlayPaused = false;\n\n    els.compareResult.innerHTML =\n      'Klik <strong>Bandingkan Node</strong> untuk melihat selisih efisiensi Minimax vs Alpha-Beta.';\n\n    renderAll(false);\n\n    if (state.gameMode === 'bvb') {\n      startAutoPlayLoop();\n    } else if (state.currentPlayer === AI) {\n      scheduleBotTurn();\n    }\n  }\n\n  function startNewMatchFromMenu() {\n    syncMenuUI();\n    switchView('game');\n    resetGame();\n  }\n\n  function goToHome() {\n    state.matchGeneration += 1;\n    clearTimers();\n    switchView('home');\n  }\n\n  function goToSetup() {\n    state.matchGeneration += 1;\n    clearTimers();\n    syncMenuUI();\n    switchView('setup');\n  }\n\n  function undoMove() {\n    if (state.history.length === 0 || state.isThinking) return;\n    state.matchGeneration += 1;\n    clearTimers();\n\n    if (state.gameMode === 'hvb') {\n      let targetIdx = state.history.length - 1;\n      while (targetIdx > 0 && state.history[targetIdx].currentPlayer !== HUMAN) {\n        targetIdx--;\n      }\n      const snap = state.history[targetIdx];\n      state.history = state.history.slice(0, targetIdx);\n      restoreSnapshot(snap);\n      renderAll(false);\n      if (state.currentPlayer === AI && !state.gameOver) {\n        scheduleBotTurn();\n      }\n    } else {\n      state.autoPlayPaused = true;\n      const snap = state.history.pop();\n      restoreSnapshot(snap);\n      renderAll(false);\n    }\n  }\n\n  function toggleAutoPlay() {\n    if (state.gameMode !== 'bvb' || state.gameOver) return;\n    if (state.autoPlayPaused) {\n      startAutoPlayLoop();\n    } else {\n      state.autoPlayPaused = true;\n      if (state.autoPlayTimer) {\n        clearTimeout(state.autoPlayTimer);\n        state.autoPlayTimer = null;\n      }\n      renderArenaHeaderAndStatus();\n    }\n  }\n\n  async function runPostGameComparison() {\n    const target = state.compareTarget;\n    if (!target) return;\n    const depth = Number(els.compareDepthSelect.value);\n    els.compareBtn.disabled = true;\n    els.compareResult.innerHTML = 'Menghitung perbandingan Minimax vs Alpha-Beta...';\n    const comp = await requestCompare(target.board, depth, target.aiPiece, target.starCells || []);\n    els.compareBtn.disabled = false;\n\n    els.compareResult.innerHTML = `\n      <table class=\"compare-table\">\n        <thead>\n          <tr>\n            <th>Algoritma (Depth ${comp.maxDepth})</th>\n            <th>Kolom</th>\n            <th>Skor Eval</th>\n            <th>Node</th>\n            <th>Waktu</th>\n          </tr>\n        </thead>\n        <tbody>\n          <tr>\n            <td>Depth-Limited Minimax</td>\n            <td>Kol ${comp.pureMinimax.bestCol}</td>\n            <td>${formatScore(comp.pureMinimax.bestScore)}</td>\n            <td>${comp.pureMinimax.nodes.toLocaleString('id-ID')}</td>\n            <td>${comp.pureMinimax.elapsedMs} ms</td>\n          </tr>\n          <tr>\n            <td>Alpha-Beta Pruning</td>\n            <td>Kol ${comp.alphaBeta.bestCol}</td>\n            <td>${formatScore(comp.alphaBeta.bestScore)}</td>\n            <td>${comp.alphaBeta.nodes.toLocaleString('id-ID')}</td>\n            <td>${comp.alphaBeta.elapsedMs} ms</td>\n          </tr>\n        </tbody>\n      </table>\n      <div class=\"compare-summary\">\n        Alpha-Beta memangkas <strong>${comp.reductionPct}%</strong> node pencarian (${(comp.pureMinimax.nodes - comp.alphaBeta.nodes).toLocaleString('id-ID')} node dihemat) dengan keputusan ${comp.sameDecision ? '100% IDENTIK' : 'berbeda'}.\n      </div>\n    `;\n  }\n\n  function bindEvents() {\n    els.modeChoiceGrid.addEventListener('click', (e) => {\n      const btn = e.target.closest('.choice-card');\n      if (!btn) return;\n      state.gameMode = btn.dataset.mode;\n      syncMenuUI();\n      pulsePickedCard(btn);\n    });\n\n    els.boardModeGrid.addEventListener('click', (e) => {\n      const btn = e.target.closest('.choice-card');\n      if (!btn) return;\n      state.boardMode = btn.dataset.bmode;\n      syncMenuUI();\n      pulsePickedCard(btn);\n    });\n\n    if (els.boardSizeGrid) {\n      els.boardSizeGrid.addEventListener('click', (e) => {\n        const btn = e.target.closest('.size-btn');\n        if (!btn) return;\n        state.boardSize = btn.dataset.size;\n        syncMenuUI();\n        pulsePickedCard(btn);\n      });\n    }\n\n    els.tileShapeGrid.addEventListener('click', (e) => {\n      const btn = e.target.closest('.shape-btn');\n      if (!btn) return;\n      state.tileShape = btn.dataset.shape;\n      syncMenuUI();\n    });\n\n    els.diffChoiceGrid.addEventListener('click', (e) => {\n      const btn = e.target.closest('.choice-card');\n      if (!btn) return;\n      state.difficulty = btn.dataset.diff;\n      syncMenuUI();\n      pulsePickedCard(btn);\n    });\n\n    els.firstChoiceGrid.addEventListener('click', (e) => {\n      const btn = e.target.closest('.choice-card');\n      if (!btn) return;\n      state.firstChoice = btn.dataset.first;\n      syncMenuUI();\n      pulsePickedCard(btn);\n    });\n\n    els.algorithmSelect.addEventListener('change', syncMenuUI);\n    els.depthSelect.addEventListener('change', syncMenuUI);\n\n    els.goSetupBtn.addEventListener('click', goToSetup);\n    els.backHomeBtn.addEventListener('click', goToHome);\n    els.startGameBtn.addEventListener('click', startNewMatchFromMenu);\n    els.backMenuBtn.addEventListener('click', goToHome);\n    els.backSetupBtn.addEventListener('click', goToSetup);\n    els.postMenuBtn.addEventListener('click', goToHome);\n    els.playAgainBtn.addEventListener('click', resetGame);\n    els.resetBtn.addEventListener('click', resetGame);\n    els.undoBtn.addEventListener('click', undoMove);\n\n    els.botStepBtn.addEventListener('click', () => {\n      if (state.gameMode === 'bvb' && !state.gameOver && !state.isThinking) {\n        state.autoPlayPaused = true;\n        if (state.autoPlayTimer) {\n          clearTimeout(state.autoPlayTimer);\n          state.autoPlayTimer = null;\n        }\n        runSingleBotStep(state.matchGeneration);\n      }\n    });\n    els.autoPlayBtn.addEventListener('click', toggleAutoPlay);\n    els.compareBtn.addEventListener('click', runPostGameComparison);\n\n    els.openRulesBtn.addEventListener('click', () => openModal(els.rulesModal));\n    els.closeRulesBtn.addEventListener('click', () => closeModal(els.rulesModal));\n    els.rulesModal.addEventListener('click', (e) => {\n      if (e.target === els.rulesModal) closeModal(els.rulesModal);\n    });\n\n    els.openInfoBtn.addEventListener('click', () => {\n      renderScoringLegend(els.modalLegendList, null, null, true);\n      openModal(els.infoModal);\n    });\n    els.closeInfoBtn.addEventListener('click', () => closeModal(els.infoModal));\n    els.infoModal.addEventListener('click', (e) => {\n      if (e.target === els.infoModal) closeModal(els.infoModal);\n    });\n\n    document.addEventListener('keydown', (e) => {\n      if (e.key === 'Escape') {\n        if (els.rulesModal.style.display === 'flex') {\n          closeModal(els.rulesModal);\n          return;\n        }\n        if (els.infoModal.style.display === 'flex') {\n          closeModal(els.infoModal);\n          return;\n        }\n      }\n      if (state.view !== 'game' || e.target.tagName === 'SELECT' || e.target.tagName === 'INPUT') return;\n      const maxColKey = String(state.board[0].length - 1);\n      if (e.key >= '0' && e.key <= maxColKey) {\n        handleHumanMove(Number(e.key));\n      }\n    });\n  }\n\n  syncMenuUI();\n  bindEvents();\n  switchView('home');\n})();\n\n</script>"


def _colab_bot_move(board_json, difficulty="Hard", ai_piece=2, use_pruning=None, custom_depth=None, star_cells_json="[]"):
    from IPython.display import JSON
    board = json.loads(board_json)
    star_cells = json.loads(star_cells_json) if star_cells_json else []
    depth_arg = int(custom_depth) if custom_depth is not None else None
    res = choose_move(
        board,
        difficulty=difficulty,
        ai_piece=int(ai_piece),
        use_pruning=use_pruning,
        custom_depth=depth_arg,
        star_cells=star_cells,
    )
    return JSON({
        "bestCol": int(res["best_col"]),
        "bestScore": int(res["best_score"]),
        "columnScores": {str(k): (int(v) if v is not None else None) for k, v in res["column_scores"].items()},
        "nodes": int(res["nodes"]),
        "elapsedMs": float(res["elapsed_ms"]),
        "maxDepth": int(res["max_depth"]),
        "usePruning": bool(res["use_pruning"]),
        "isRandom": bool(res.get("is_random", False)),
    })


def _colab_compare(board_json, max_depth=4, ai_piece=2, star_cells_json="[]"):
    from IPython.display import JSON
    board = json.loads(board_json)
    star_cells = json.loads(star_cells_json) if star_cells_json else []
    comp = compare_algorithms(board, max_depth=int(max_depth), ai_piece=int(ai_piece), star_cells=star_cells)
    return JSON({
        "maxDepth": int(comp["max_depth"]),
        "reductionPct": float(comp["reduction_pct"]),
        "sameDecision": bool(comp["same_decision"]),
        "pureMinimax": {
            "bestCol": int(comp["pure"]["best_col"]),
            "bestScore": int(comp["pure"]["best_score"]),
            "nodes": int(comp["pure"]["nodes"]),
            "elapsedMs": float(comp["pure"]["elapsed_ms"]),
        },
        "alphaBeta": {
            "bestCol": int(comp["alphabeta"]["best_col"]),
            "bestScore": int(comp["alphabeta"]["best_score"]),
            "nodes": int(comp["alphabeta"]["nodes"]),
            "elapsedMs": float(comp["alphabeta"]["elapsed_ms"]),
        },
    })


def launch_retro_colab_ui():
    """Render UI Retro 3-Layar terhubung langsung ke fungsi Python di Google Colab."""
    if "choose_move" not in globals():
        print("Jalankan semua sel di atas terlebih dahulu (Runtime -> Run all / Jalankan semua)!")
        return None

    try:
        from IPython.display import HTML, display
    except ImportError:
        print("Info: IPython tidak tersedia di environment ini. Gunakan play_game() pada Bagian 6A untuk mode konsol.")
        return None

    try:
        from google.colab import output
        output.register_callback("connect4.bot_move", _colab_bot_move)
        output.register_callback("connect4.compare", _colab_compare)
        print("[Colab Python Kernel Terhubung] Bot AI & Benchmark di UI mengeksekusi Python runtime Colab.")
    except ImportError:
        print("[Mode Notebook Lokal] google.colab tidak ditemukan; UI berjalan dengan fallback JS 1:1.")

    display(HTML(RETRO_UI_HTML))
    return True


launch_retro_colab_ui()